# Stage 06 — Fingerprint candidate ranking

Bridge Stage 05 molecule-level predictions and the Stage 04 mass-sorted database.
Run all cells in order on Kaggle with Stage 02/04/05 outputs attached. CPU is
sufficient; no network access is needed. Sampled tuning uses **val_model only**;
val_rank is held out from this scorer's tuning for Stage 09 (not cross-validated
out-of-fold neural predictions). Stage 05 selects its model on val_model;
Stage 06 also reserves val_rank labels until after freezing its configuration.

Outputs: three top-100 feature tables, a validated top-25 submission, frozen
config, manifest, and `/kaggle/working/fprank.py` for Stage 10.

In [ ]:
!pip install -q rdkit
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from pathlib import Path
import importlib.util, sys, json, hashlib, time, gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
WORK = Path('/kaggle/working')
OUT = WORK / 'artifacts/fprank'
OUT.mkdir(parents=True, exist_ok=True)
STARTED = time.time()
# Optional overrides; discovery prefers working files, then newest mtime.
DB_DIR = None
PRED_DIR = None
DATA_DIR = None
SPLIT_DIR = None

## Stage and import modules
Embedded snapshots make this notebook self-contained. All four modules are
written to and imported strictly from `/kaggle/working/`. The candidate module
checks the database fingerprint schema against evalkit before retrieval.

In [ ]:
(WORK / 'evalkit.py').write_text('"""\nevalkit.py - Master Cheminformatics & Metric Toolkit for Enveda CASMI 2026.\nGenerated by Stage 02 (02_eval_splits.ipynb).\nSingle source of truth for all downstream stages (models, retrieval, reranker, submission).\n"""\nimport os\nimport sys\nimport time\nimport math\nimport warnings\nimport functools\nfrom typing import Dict, List, Tuple, Optional, Union, Any, Set\nimport numpy as np\nimport pandas as pd\n\n# RDKit imports\ntry:\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem\n    from rdkit.Chem.Scaffolds import MurckoScaffold\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    try:\n        from rdkit.Chem import rdFingerprintGenerator\n        _HAS_FP_GEN = True\n    except ImportError:\n        _HAS_FP_GEN = False\n    RDLogger.DisableLog("rdApp.*")\n    RDKIT_VERSION = Chem.rdBase.rdkitVersion\nexcept ImportError as e:\n    raise ImportError(f"RDKit is mandatory for CASMI 2026 evalkit: {e}")\n\n# Version verification\nOFFICIAL_RDKIT_VERSION = "2026.03.3"\nif RDKIT_VERSION != OFFICIAL_RDKIT_VERSION:\n    warnings.warn(\n        f"[evalkit] Official competition scorer pins RDKit {OFFICIAL_RDKIT_VERSION}; "\n        f"current environment is {RDKIT_VERSION}. Tautomer canonicalization for rare "\n        f"conjugated systems may produce minor edge-case discrepancies.",\n        UserWarning\n    )\n\n# ── 1. Shared Fingerprint Specification ─────────────────────────────────────\nFP_SPEC: Dict[str, Any] = {\n    "morgan": {\n        "radius": 2,\n        "nBits": 4096,\n        "chirality": False,\n        "dtype": "uint8",\n    },\n    "maccs": {\n        "nBits": 167,\n        "dtype": "uint8",\n    },\n}\n\n# ── 2. Adduct Carrier Mass Invariant ─────────────────────────────────────────\nCARRIER_MASSES: Dict[str, float] = {\n    "[M+H]+":      1.007276,\n    "[M-H]-":     -1.007276,\n    "[M+Na]+":    22.989218,\n    "[M+NH4]+":   18.033823,\n    "[M+K]+":     38.963158,\n    "[M+H-H2O]+": -17.002740,\n    "[M+FA-H]-":  44.998201,\n    "[M+Cl]-":    34.969402,\n}\n\ndef calc_neutral_mass(precursor_mz: float, adduct: str, polarity: str = "+") -> float:\n    """\n    Strictly compute neutral mass as: M_neutral = (precursor_mz * |z|) - carrier_mass\n    Fallback: [M+H]+ for positive polarity, [M-H]- for negative polarity.\n    """\n    add_clean = str(adduct).strip() if adduct is not None else ""\n    if add_clean in CARRIER_MASSES:\n        carrier = CARRIER_MASSES[add_clean]\n    else:\n        pol_str = str(polarity).strip().lower()\n        if pol_str in ["positive", "pos", "+", "1"]:\n            carrier = CARRIER_MASSES["[M+H]+"]\n        else:\n            carrier = CARRIER_MASSES["[M-H]-"]\n    return float(precursor_mz) - carrier\n\n# ── 3. Molecule Standardization & Desalting ─────────────────────────────────\ndef standardize_mol(mol_or_smiles: Union[str, Chem.Mol]) -> Optional[Chem.Mol]:\n    """\n    Standardize a molecule for downstream evaluation and library indexing:\n    - Retain largest organic fragment (heavy atoms >= 3).\n    - Neutralize formal charges via rdMolStandardize.Uncharger.\n    - Clear isotope labels.\n    - Sanitize.\n    - Filter heavy atoms in [3, 120].\n    """\n    if isinstance(mol_or_smiles, str):\n        if not mol_or_smiles or mol_or_smiles.strip() == "":\n            return None\n        mol = Chem.MolFromSmiles(mol_or_smiles)\n    else:\n        mol = mol_or_smiles\n    if mol is None:\n        return None\n    try:\n        # Desalt: Largest fragment chooser\n        lfc = rdMolStandardize.LargestFragmentChooser()\n        mol = lfc.choose(mol)\n        if mol is None:\n            return None\n        n_heavy = mol.GetNumHeavyAtoms()\n        if n_heavy < 3 or n_heavy > 120:\n            return None\n        # Neutralize charges\n        uncharger = rdMolStandardize.Uncharger()\n        mol = uncharger.uncharge(mol)\n        if mol is None:\n            return None\n        # Clear isotopes\n        for atom in mol.GetAtoms():\n            if atom.GetIsotope() != 0:\n                atom.SetIsotope(0)\n        # Sanitize\n        Chem.SanitizeMol(mol)\n        return mol\n    except Exception:\n        return None\n\n# ── 4. InChIKey Generation & Speed Guarded Canonicalization ───────────────────\n_IK14_CACHE: Dict[str, str] = {}\n_FAST_IK14_CACHE: Dict[str, str] = {}\n\ndef fast_key(mol_or_smiles: Union[str, Chem.Mol]) -> str:\n    """\n    Compute plain InChIKey14 without tautomer enumeration.\n    Extremely fast for candidate filtering and initial deduplication.\n    """\n    if isinstance(mol_or_smiles, str):\n        if mol_or_smiles in _FAST_IK14_CACHE:\n            return _FAST_IK14_CACHE[mol_or_smiles]\n        smi = mol_or_smiles\n        mol = Chem.MolFromSmiles(smi)\n    else:\n        smi = None\n        mol = mol_or_smiles\n    if mol is None:\n        return ""\n    try:\n        ik14 = Chem.MolToInchiKey(mol)[:14]\n    except Exception:\n        ik14 = ""\n    if smi is not None:\n        _FAST_IK14_CACHE[smi] = ik14\n    return ik14\n\n# Tautomer enumerator singleton with speed guards\n_TAUTOMER_ENUMERATOR = rdMolStandardize.TautomerEnumerator()\n_TAUTOMER_ENUMERATOR.SetMaxTautomers(50)\n_TAUTOMER_ENUMERATOR.SetMaxTransforms(100)\n\ndef canon_ik14(mol_or_smiles: Union[str, Chem.Mol], timeout_sec: float = 1.5) -> str:\n    """\n    Compute official ground-truth canonical InChIKey14:\n    - RDKit TautomerEnumerator canonicalization\n    - Max tautomers = 50, max transforms = 100\n    - Fallback to fast_key on timeout or error\n    - In-memory cached\n    """\n    if isinstance(mol_or_smiles, str):\n        smi = mol_or_smiles.strip()\n        if smi in _IK14_CACHE:\n            return _IK14_CACHE[smi]\n        mol = Chem.MolFromSmiles(smi)\n    else:\n        smi = None\n        mol = mol_or_smiles\n    if mol is None:\n        return ""\n    # Speed guarded execution\n    t0 = time.time()\n    try:\n        canon_mol = _TAUTOMER_ENUMERATOR.Canonicalize(mol)\n        if time.time() - t0 > timeout_sec:\n            ik14 = Chem.MolToInchiKey(mol)[:14]\n        else:\n            ik14 = Chem.MolToInchiKey(canon_mol)[:14]\n    except Exception:\n        ik14 = Chem.MolToInchiKey(mol)[:14]\n    if smi is not None:\n        _IK14_CACHE[smi] = ik14\n    return ik14\n\n# ── 5. Fingerprint Generators ────────────────────────────────────────────────\nif _HAS_FP_GEN:\n    _MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\nelse:\n    _MORGAN_GEN = None\n\ndef compute_morgan_fp(mol_or_smiles: Union[str, Chem.Mol], n_bits: int = 4096, radius: int = 2) -> np.ndarray:\n    """Compute binary Morgan fingerprint as uint8 numpy vector of shape (n_bits,)."""\n    if isinstance(mol_or_smiles, str):\n        mol = Chem.MolFromSmiles(mol_or_smiles)\n    else:\n        mol = mol_or_smiles\n    if mol is None:\n        return np.zeros(n_bits, dtype=np.uint8)\n    if _MORGAN_GEN is not None and n_bits == 4096 and radius == 2:\n        return _MORGAN_GEN.GetFingerprintAsNumPy(mol).astype(np.uint8)\n    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits, useChirality=False)\n    arr = np.zeros(n_bits, dtype=np.uint8)\n    for bit in fp.GetOnBits():\n        arr[bit] = 1\n    return arr\n\ndef compute_maccs_fp(mol_or_smiles: Union[str, Chem.Mol]) -> np.ndarray:\n    """Compute binary MACCS keys fingerprint as uint8 numpy vector of shape (167,)."""\n    if isinstance(mol_or_smiles, str):\n        mol = Chem.MolFromSmiles(mol_or_smiles)\n    else:\n        mol = mol_or_smiles\n    if mol is None:\n        return np.zeros(167, dtype=np.uint8)\n    fp = rdMolDescriptors.GetMACCSKeysFingerprint(mol)\n    arr = np.zeros(167, dtype=np.uint8)\n    for bit in fp.GetOnBits():\n        arr[bit] = 1\n    return arr\n\n# ── 6. Bemis-Murcko Scaffold ─────────────────────────────────────────────────\ndef get_murcko_scaffold(mol_or_smiles: Union[str, Chem.Mol]) -> str:\n    """Compute Bemis-Murcko scaffold SMILES without chirality, with acyclic formula fallback."""\n    if isinstance(mol_or_smiles, str):\n        mol = Chem.MolFromSmiles(mol_or_smiles)\n    else:\n        mol = mol_or_smiles\n    if mol is None:\n        return ""\n    try:\n        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)\n        if scaf and len(scaf) > 0:\n            return scaf\n        form = rdMolDescriptors.CalcMolFormula(mol)\n        return f"_acyclic_{form}"\n    except Exception:\n        return "_unknown"\n\n# ── 7. Competition Metric: MRR@25 ────────────────────────────────────────────\ndef mrr_at_25(pred: Dict[str, List[str]], truth: Dict[str, str]) -> float:\n    """\n    Exact local replica of official CASMI 2026 MRR@25 competition metric.\n    - Evaluates raw order of predictions, top 25 guesses per molecule_id.\n    - Reciprocal rank = 1.0 / pos of first guess whose canon_ik14 equals truth\'s canon_ik14.\n    - Returns mean reciprocal rank across all ground truth query IDs.\n    """\n    scores: List[float] = []\n    for qid, t_val in truth.items():\n        t_key = t_val if (len(t_val) == 14 and t_val.isupper()) else canon_ik14(t_val)\n        guesses = pred.get(qid, [])[:25]\n        rr = 0.0\n        for i, g in enumerate(guesses, start=1):\n            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n            if g_key == t_key:\n                rr = 1.0 / i\n                break\n        scores.append(rr)\n    return float(np.mean(scores)) if len(scores) > 0 else 0.0\n\ndef mrr_at_25_dedup(pred: Dict[str, List[str]], truth: Dict[str, str]) -> Tuple[float, float, int]:\n    """\n    Diagnostic variant evaluating MRR@25 after InChIKey14 deduplication.\n    Returns:\n        (mrr_raw, mrr_dedup, total_wasted_ranks)\n    """\n    raw_scores: List[float] = []\n    dedup_scores: List[float] = []\n    total_wasted = 0\n    for qid, t_val in truth.items():\n        t_key = t_val if (len(t_val) == 14 and t_val.isupper()) else canon_ik14(t_val)\n        raw_guesses = pred.get(qid, [])[:25]\n        # Raw MRR\n        rr_raw = 0.0\n        for i, g in enumerate(raw_guesses, start=1):\n            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n            if g_key == t_key:\n                rr_raw = 1.0 / i\n                break\n        raw_scores.append(rr_raw)\n        # Deduplicate guesses by InChIKey14 while preserving order\n        seen_keys: Set[str] = set()\n        dedup_guesses: List[str] = []\n        for g in raw_guesses:\n            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n            if g_key not in seen_keys:\n                seen_keys.add(g_key)\n                dedup_guesses.append(g)\n        total_wasted += (len(raw_guesses) - len(dedup_guesses))\n        # Dedup MRR\n        rr_dedup = 0.0\n        for i, g in enumerate(dedup_guesses[:25], start=1):\n            g_key = g if (len(g) == 14 and g.isupper()) else canon_ik14(g)\n            if g_key == t_key:\n                rr_dedup = 1.0 / i\n                break\n        dedup_scores.append(rr_dedup)\n    mrr_raw = float(np.mean(raw_scores)) if raw_scores else 0.0\n    mrr_dedup = float(np.mean(dedup_scores)) if dedup_scores else 0.0\n    return mrr_raw, mrr_dedup, total_wasted\n\n# ── 8. Submission Validator ──────────────────────────────────────────────────\ndef validate_submission(\n    df_or_path: Union[pd.DataFrame, str],\n    test_ids: Union[List[str], Set[str], pd.Series]\n) -> Tuple[bool, List[str], List[str], Dict[str, Any]]:\n    """\n    Rigorous validator implementing all competition rejection rules:\n    - Column contract: [\'molecule_id\', \'smiles\']\n    - 100% test ID completeness (zero missing, zero duplicates)\n    - Non-null, non-empty SMILES string\n    - Up to 25 guesses per query joined by \';\'\n    - RDKit parseability of every individual SMILES guess\n    - Detection of internal duplicate InChIKey14s (warning)\n    Returns:\n        (is_valid, errors, warnings, summary_stats)\n    """\n    errors: List[str] = []\n    warnings_list: List[str] = []\n    summary: Dict[str, Any] = {}\n\n    if isinstance(df_or_path, str):\n        if not os.path.exists(df_or_path):\n            return False, [f"Submission file does not exist: {df_or_path}"], [], {}\n        df = pd.read_csv(df_or_path)\n    else:\n        df = df_or_path.copy()\n\n    # Column contract\n    expected_cols = {"molecule_id", "smiles"}\n    if not expected_cols.issubset(set(df.columns)):\n        missing = expected_cols - set(df.columns)\n        errors.append(f"Missing required columns: {missing}. Found: {list(df.columns)}")\n        return False, errors, warnings_list, summary\n\n    # Row completeness\n    test_set = set(str(x) for x in test_ids)\n    sub_set = set(df["molecule_id"].astype(str))\n    if len(df) != len(test_set):\n        errors.append(f"Row count mismatch: submission has {len(df)} rows, expected {len(test_set)}")\n    missing_ids = test_set - sub_set\n    if len(missing_ids) > 0:\n        errors.append(f"Missing {len(missing_ids)} test molecule_ids! Example missing: {list(missing_ids)[:5]}")\n    extra_ids = sub_set - test_set\n    if len(extra_ids) > 0:\n        errors.append(f"Found {len(extra_ids)} unexpected molecule_ids not in test set!")\n    if df["molecule_id"].duplicated().any():\n        n_dups = int(df["molecule_id"].duplicated().sum())\n        errors.append(f"Duplicate molecule_id entries found in submission: {n_dups} duplicates")\n\n    # Content checks\n    null_smiles = int(df["smiles"].isna().sum())\n    if null_smiles > 0:\n        errors.append(f"Found {null_smiles} null/NaN values in \'smiles\' column")\n\n    total_guesses = 0\n    total_unparseable = 0\n    total_internal_dups = 0\n    max_guesses_seen = 0\n\n    for idx, row in df.iterrows():\n        s_val = str(row["smiles"]) if pd.notna(row["smiles"]) else ""\n        if len(s_val.strip()) == 0:\n            errors.append(f"Empty SMILES string for molecule_id \'{row[\'molecule_id\']}\' at row {idx}")\n            continue\n        parts = [p.strip() for p in s_val.split(";") if p.strip()]\n        n_p = len(parts)\n        max_guesses_seen = max(max_guesses_seen, n_p)\n        total_guesses += n_p\n        if n_p > 25:\n            errors.append(f"Row {idx} (\'{row[\'molecule_id\']}\') has {n_p} guesses; maximum permitted is 25")\n        # Check parseability and duplicates\n        seen_keys: Set[str] = set()\n        for p in parts:\n            mol = Chem.MolFromSmiles(p)\n            if mol is None:\n                total_unparseable += 1\n                if total_unparseable <= 5:\n                    errors.append(f"Invalid unparseable SMILES \'{p}\' in row {idx} (\'{row[\'molecule_id\']}\')")\n            else:\n                k = fast_key(mol)\n                if k in seen_keys:\n                    total_internal_dups += 1\n                seen_keys.add(k)\n\n    if total_unparseable > 5:\n        errors.append(f"Total unparseable SMILES across submission: {total_unparseable}")\n    if total_internal_dups > 0:\n        warnings_list.append(\n            f"Found {total_internal_dups} duplicate InChIKey14 guesses inside prediction lists! "\n            f"These waste ranking positions without improving MRR@25. Strictly deduplicate before saving."\n        )\n\n    is_valid = (len(errors) == 0)\n    summary = {\n        "is_valid": is_valid,\n        "n_rows": len(df),\n        "expected_rows": len(test_set),\n        "total_guesses": total_guesses,\n        "avg_guesses_per_mol": total_guesses / max(len(df), 1),\n        "max_guesses_seen": max_guesses_seen,\n        "total_unparseable": total_unparseable,\n        "total_internal_dups": total_internal_dups,\n        "n_errors": len(errors),\n        "n_warnings": len(warnings_list),\n    }\n    return is_valid, errors, warnings_list, summary\n', encoding='utf-8')

In [ ]:
(WORK / 'candidates.py').write_text('"""Disk-backed candidate retrieval. Import only from /kaggle/working.\n\nCandidate pools uniformly use fast_key, standardized with standardize_mol().\nUse finalize_candidates AFTER ranking to compute canon_ik14 only on the filtered\ntop-256 candidates per query for tautomer-canonical deduplication.\nRow IDs are stable positions in db.parquet and all row-aligned .npy files.\n"""\nimport csv\nimport gc\nimport gzip\nimport hashlib\nimport heapq\nimport importlib.util\nimport io\nimport itertools\nimport json\nimport math\nimport multiprocessing as mp\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport shutil\nimport sqlite3\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter, OrderedDict\nfrom dataclasses import asdict\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport psutil\nfrom rdkit import Chem, DataStructs, RDLogger\nfrom rdkit.Chem import MACCSkeys, rdMolDescriptors, rdFingerprintGenerator\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\nfrom tqdm.auto import tqdm\n\nRDLogger.DisableLog("rdApp.*")\nWORK = Path("/kaggle/working")\n_max_csv = sys.maxsize\nwhile True:\n    try:\n        csv.field_size_limit(_max_csv)\n        break\n    except OverflowError:\n        _max_csv = int(_max_csv / 10)\nBASE_ELEMENTS = {"C", "H", "N", "O", "S", "P", "F", "Cl", "Br", "I"}\nEXTRA_ELEMENTS = {"Na", "K", "Ca", "Li", "Mg", "Fe", "Zn", "Co", "Cu", "Mn", "Mo", "Ni", "V", "W", "B", "Si", "Se", "As"}\nTRAIN_BIT = 1\nDB_COLUMNS = ["row_id", "ik14", "smiles", "formula", "mass",\n              "source_mask", "fast_key", "identity_kind"]\nCTX = None\nEK = None\n_MEM_IK14_CACHE = OrderedDict()\n_MEM_FP_CACHE = OrderedDict()\n\n\ndef find_file(filename, hint="", required=True, roots=None):\n    if filename.endswith(".py"):\n        paths = [WORK / filename] if (WORK / filename).is_file() else []\n    else:\n        roots = roots or [WORK, Path("/kaggle/input")]\n        paths = list({p.resolve() for root in roots for p in Path(root).rglob(filename)})\n    paths.sort(key=lambda p: (bool(hint and hint in str(p)), p.stat().st_mtime,\n                              str(p)), reverse=True)\n    if paths:\n        print(f"[find_file] {filename}: {paths[0]}")\n        return paths[0]\n    if required:\n        raise FileNotFoundError(f"Missing {filename}; modules must be in {WORK}")\n    print(f"[find_file] optional {filename}: absent")\n    return None\n\n\ndef find_manifest(stage):\n    matches = []\n    for root in (WORK, Path("/kaggle/input")):\n        for path in root.rglob("manifest*.json"):\n            try:\n                record = json.loads(path.read_text())\n            except (OSError, ValueError) as exc:\n                print(f"Unreadable manifest {path}: {exc}")\n                continue\n            if record.get("stage") == stage:\n                matches.append((path.stat().st_mtime, path, record))\n    return max(matches, default=(None, None, None), key=lambda x: x[0])[1:]\n\n\ndef load_evalkit():\n    global EK\n    if EK is None:\n        path = find_file("evalkit.py")\n        if path.resolve() != (WORK / "evalkit.py").resolve():\n            raise RuntimeError("evalkit.py must be read only from /kaggle/working")\n        spec = importlib.util.spec_from_file_location("evalkit", path)\n        module = importlib.util.module_from_spec(spec)\n        sys.modules["evalkit"] = module\n        spec.loader.exec_module(module)\n        for name in ("FP_SPEC", "fast_key", "canon_ik14", "mrr_at_25"):\n            if not hasattr(module, name):\n                raise AttributeError(f"evalkit.py is missing {name}")\n        EK = module\n    return EK\n\n\ndef open_sql(path, readonly=False):\n    uri = Path(path).resolve().as_uri() + "?mode=ro"\n    db = sqlite3.connect(uri if readonly else str(path), uri=readonly, timeout=90)\n    db.execute("PRAGMA cache_size=-65536")\n    db.execute("PRAGMA temp_store=FILE")\n    db.execute("PRAGMA mmap_size=0")\n    if not readonly:\n        db.execute("PRAGMA journal_mode=DELETE")\n    return db\n\n\ndef resource_guard(cfg, reserve_bytes=0):\n    proc = psutil.Process()\n    rss = proc.memory_info().rss\n    for child in proc.children(recursive=True):\n        try:\n            rss += child.memory_info().rss\n        except psutil.NoSuchProcess:\n            pass\n    if rss + reserve_bytes > cfg.RAM_GB * 1e9:\n        raise MemoryError(f"RAM guard: {rss/1e9:.2f} GB exceeds {cfg.RAM_GB} GB")\n    free = shutil.disk_usage(cfg.output_dir).free\n    if free < cfg.MIN_FREE_DISK_GB * 1e9:\n        raise OSError(f"Only {free/1e9:.2f} GB disk free; insufficient space")\n    return rss\n\n\ndef batched(iterable, size):\n    iterator = iter(iterable)\n    while True:\n        batch = list(itertools.islice(iterator, size))\n        if not batch:\n            return\n        yield batch\n\n\ndef init_worker():\n    os.environ["OMP_NUM_THREADS"] = "1"\n    os.environ["OPENBLAS_NUM_THREADS"] = "1"\n    pa.set_cpu_count(1)\n    pa.set_io_thread_count(1)\n    load_evalkit()\n\n\ndef pool_map(pool, function, iterable, cfg, total=None, desc="Processing"):\n    with tqdm(total=total, desc=desc, unit="rows") as progress:\n        for wave in batched(iterable, cfg.WORKERS * cfg.CHUNK_ROWS):\n            jobs = [pool.apply_async(function, (chunk,))\n                    for chunk in batched(wave, cfg.CHUNK_ROWS)]\n            for job in jobs:\n                result = job.get()\n                yield result\n                progress.update(len(result))\n            resource_guard(cfg)\n\n\ndef standardize_mol(mol_or_smiles, allowed=None):\n    if isinstance(mol_or_smiles, str):\n        if not mol_or_smiles or not mol_or_smiles.strip():\n            return None\n        mol = Chem.MolFromSmiles(mol_or_smiles.strip())\n    else:\n        mol = mol_or_smiles\n    if mol is None:\n        return None\n    try:\n        lfc = rdMolStandardize.LargestFragmentChooser()\n        mol = lfc.choose(mol)\n        if mol is None:\n            return None\n        uncharger = rdMolStandardize.Uncharger()\n        mol = uncharger.uncharge(mol)\n        if mol is None or Chem.GetFormalCharge(mol) != 0:\n            return None\n        for atom in mol.GetAtoms():\n            if atom.GetIsotope() != 0:\n                atom.SetIsotope(0)\n        Chem.RemoveStereochemistry(mol)\n        Chem.SanitizeMol(mol)\n        n_heavy = mol.GetNumHeavyAtoms()\n        if n_heavy < 1 or n_heavy > 150:\n            return None\n        if allowed is not None:\n            atoms = {a.GetSymbol() for a in mol.GetAtoms()}\n            if not atoms.issubset(allowed):\n                return None\n        return mol\n    except Exception:\n        return None\n\n\ndef parent_mol(smiles, allowed):\n    try:\n        if not isinstance(smiles, str) or not smiles.strip():\n            return None, "invalid"\n        mol = Chem.MolFromSmiles(smiles.strip())\n        if mol is None:\n            return None, "invalid"\n        lfc = rdMolStandardize.LargestFragmentChooser()\n        mol = lfc.choose(mol)\n        if mol is None:\n            return None, "fragment_failure"\n        uncharger = rdMolStandardize.Uncharger()\n        mol = uncharger.uncharge(mol)\n        if mol is None or Chem.GetFormalCharge(mol) != 0:\n            return None, "non_neutralizable"\n        for atom in mol.GetAtoms():\n            if atom.GetIsotope() != 0:\n                atom.SetIsotope(0)\n        Chem.RemoveStereochemistry(mol)\n        Chem.SanitizeMol(mol)\n        n_heavy = mol.GetNumHeavyAtoms()\n        if n_heavy < 1 or n_heavy > 150:\n            return None, "heavy_atoms_out_of_bounds"\n        atoms = {a.GetSymbol() for a in mol.GetAtoms()}\n        if not atoms.issubset(allowed):\n            return None, "disallowed_element"\n        return mol, "ok"\n    except Exception:\n        return None, "rdkit_failure"\n\n\ndef standardize_chunk(tasks):\n    ek = load_evalkit()\n    results = []\n    for raw, source, allowed, known_ik14 in tasks:\n        mol, status = parent_mol(raw, set(allowed))\n        row = None\n        if mol is not None:\n            key = ek.fast_key(mol)\n            ik14 = known_ik14 if (known_ik14 and isinstance(known_ik14, str)) else None\n            if not key:\n                status = "key_failure"\n            else:\n                row = (key, Chem.MolToSmiles(mol, isomericSmiles=False),\n                       rdMolDescriptors.CalcMolFormula(mol),\n                       float(rdMolDescriptors.CalcExactMolWt(mol)), source, ik14,\n                       ",".join(sorted({a.GetSymbol() for a in mol.GetAtoms()})))\n        results.append((raw, status, row))\n    return results\n\n\ndef fp_layout():\n    spec = load_evalkit().FP_SPEC\n    morgan = spec["morgan"]\n    if morgan.get("chirality", False):\n        raise ValueError("FP_SPEC must use achiral Morgan fingerprints")\n    widths = [int(morgan["nBits"])]\n    if "maccs" in spec:\n        if int(spec["maccs"]["nBits"]) != 167:\n            raise ValueError("RDKit MACCS requires FP_SPEC.maccs.nBits=167")\n        widths.append(167)\n    return widths, sum((n + 7) // 8 for n in widths)\n\n\ndef fingerprint_chunk(rows):\n    spec = load_evalkit().FP_SPEC\n    widths, _ = fp_layout()\n    gen = rdFingerprintGenerator.GetMorganGenerator(\n        radius=int(spec["morgan"]["radius"]),\n        fpSize=int(spec["morgan"]["nBits"]), includeChirality=False)\n    result = []\n    for row_id, smiles in rows:\n        mol = Chem.MolFromSmiles(smiles)\n        if mol is None:\n            raise ValueError(f"Invalid standardized SMILES at row {row_id}")\n        arrays = [gen.GetFingerprintAsNumPy(mol).astype(np.uint8)]\n        if len(widths) == 2:\n            bits = np.zeros(167, dtype=np.uint8)\n            DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), bits)\n            arrays.append(bits)\n        packed = np.concatenate([np.packbits(a, bitorder="little") for a in arrays])\n        result.append((row_id, packed))\n    return result\n\n\nclass SchemaAdapter:\n    aliases = {\n        "molecule_id": ["molecule_id", "inchikey14", "id"],\n        "smiles": ["smiles", "canonical_smiles", "canonicalsmiles", "normalized_smiles",\n                   "isomeric_smiles", "isomericsmiles", "smiles_iso", "structure_smiles"],\n        "neutral_mass": ["neutral_mass", "neutral_precursor_mass"],\n        "precursor_mz": ["precursor_mz", "precursormz", "parent_mz"],\n        "adduct": ["adduct", "precursor_type"],\n        "polarity": ["polarity", "ionization_mode", "ionmode"],\n        "split3": ["split3"],\n    }\n\n    def __init__(self, columns, name, required=(), schema=None):\n        self.name = name\n        lower = {str(c).lower(): c for c in columns}\n        self.mapping = {}\n        hints = (schema or {}).get(name, {}).get("mapping", {})\n        for field, aliases in self.aliases.items():\n            choices = aliases + [hints.get(field, "")]\n            for alias in choices:\n                if alias.lower() in lower:\n                    self.mapping[field] = lower[alias.lower()]\n                    break\n        missing = set(required) - set(self.mapping)\n        if missing:\n            raise ValueError(f"SchemaAdapter({name}): missing {sorted(missing)}; columns={columns}")\n        print(f"SchemaAdapter({name}): {self.mapping}")\n\n    def adapt(self, table):\n        frame = table.to_pandas()\n        return frame.rename(columns={v: k for k, v in self.mapping.items()})\n\n\ndef observed_mass(frame):\n    mass = pd.to_numeric(frame.get("neutral_mass", pd.Series(np.nan, index=frame.index)),\n                         errors="coerce").to_numpy(dtype=np.float64)\n    missing = ~np.isfinite(mass) | (mass <= 0)\n    if not missing.any():\n        return mass\n    if "precursor_mz" not in frame:\n        raise ValueError("Missing valid observed neutral_mass and precursor_mz")\n    add = frame.get("adduct", pd.Series("", index=frame.index)).fillna("").astype(str)\n    pol = frame.get("polarity", pd.Series("+", index=frame.index)).fillna("+").astype(str)\n    table = {"[M+H]+": (1, 1.007276), "[M-H]-": (1, -1.007276),\n             "[M+Na]+": (1, 22.989218), "[M+K]+": (1, 38.963158),\n             "[M+NH4]+": (1, 18.033823), "[M+Cl]-": (1, 34.969402),\n             "[M+H-H2O]+": (1, -17.003289), "[M+FA-H]-": (1, 44.998201),\n             "[M+2H]2+": (2, 2.014552), "[M-2H]2-": (2, -2.014552)}\n    negative = pol.str.lower().isin(["negative", "neg", "-", "-1"])\n    charge = add.map({k: v[0] for k, v in table.items()}).fillna(1).to_numpy()\n    shift = add.map({k: v[1] for k, v in table.items()}).to_numpy()\n    shift = np.where(np.isfinite(shift), shift, np.where(negative, -1.007276, 1.007276))\n    mz = pd.to_numeric(frame.precursor_mz, errors="coerce").to_numpy(dtype=np.float64)\n    mass[missing] = (mz * charge - shift)[missing]\n    if (~np.isfinite(mass) | (mass <= 0)).any():\n        raise ValueError("Non-positive/invalid observed neutral masses after adduct fallback")\n    return mass\n\n\ndef initialize_stage(cfg):\n    path = Path(cfg.output_dir) / "build.sqlite"\n    db = open_sql(path)\n    db.executescript("""\n    CREATE TABLE IF NOT EXISTS state(k TEXT PRIMARY KEY, v TEXT);\n    CREATE TABLE IF NOT EXISTS molecules(id TEXT PRIMARY KEY, smiles TEXT,\n        mass_sum REAL, n INTEGER, lo REAL, hi REAL, split3 TEXT, ik14 TEXT);\n    CREATE TABLE IF NOT EXISTS tests(id TEXT PRIMARY KEY, smiles TEXT,\n        mass_sum REAL, n INTEGER, lo REAL, hi REAL, split3 TEXT);\n    CREATE TABLE IF NOT EXISTS splits(id TEXT PRIMARY KEY, split3 TEXT, ik14 TEXT);\n    CREATE TABLE IF NOT EXISTS train_structures(raw TEXT PRIMARY KEY,\n        fast_key TEXT, ik14 TEXT, smiles TEXT, formula TEXT, mass REAL, status TEXT);\n    CREATE TABLE IF NOT EXISTS structures(fast_key TEXT PRIMARY KEY, smiles TEXT,\n        formula TEXT, mass REAL, source_mask INTEGER, ik14 TEXT, elements TEXT);\n    CREATE TABLE IF NOT EXISTS sources(path TEXT PRIMARY KEY, done INTEGER);\n    """)\n    return db\n\n\ndef state_get(db, key, default=None):\n    row = db.execute("SELECT v FROM state WHERE k=?", (key,)).fetchone()\n    return json.loads(row[0]) if row else default\n\n\ndef state_set(db, key, value):\n    db.execute("INSERT OR REPLACE INTO state VALUES (?,?)", (key, json.dumps(value)))\n    db.commit()\n\n\ndef ingest_metadata(db, path, kind, cfg, schema=None):\n    if state_get(db, f"metadata_{kind}"):\n        return\n    table = "molecules" if kind == "train" else "tests"\n    db.execute(f"DELETE FROM {table}")\n    file = pq.ParquetFile(path)\n    required = ("molecule_id", "smiles") if kind == "train" else ("molecule_id",)\n    adapter = SchemaAdapter(file.schema_arrow.names, kind, required, schema)\n    columns = [v for k, v in adapter.mapping.items() if k not in ("smiles", "split3")]\n    if kind == "train":\n        columns.append(adapter.mapping["smiles"])\n    table_arrow = file.read(columns=columns)\n    frame = adapter.adapt(table_arrow)\n    if frame.molecule_id.isna().any():\n        raise ValueError(f"Null molecule_id in {path}")\n    frame["id"] = frame.molecule_id.astype(str)\n    frame["mass"] = observed_mass(frame)\n    if kind == "train":\n        frame["smiles"] = frame.smiles.fillna("").astype(str)\n        valid = frame[frame.smiles != ""]\n        if not valid.empty:\n            counts = valid.groupby(["id", "smiles"], sort=False).size().reset_index(name="c")\n            consensus = counts.sort_values(["id", "c"], ascending=[True, False]).drop_duplicates(subset=["id"])\n            smi_map = consensus.set_index("id")["smiles"]\n        else:\n            smi_map = pd.Series(dtype=str)\n    groups = frame.groupby("id", sort=False)\n    grouped = groups.agg(mass_median=("mass", "median"),\n                         n=("mass", "size"), lo=("mass", "min"), hi=("mass", "max"))\n    # Trick SQLite: Store (median * n) so that SQL \'mass_sum / n\' perfectly yields the median\n    grouped["mass_sum"] = grouped["mass_median"] * grouped["n"]\n    if kind == "train":\n        grouped["smiles"] = smi_map.reindex(grouped.index).fillna(groups["smiles"].first()).fillna("")\n    else:\n        grouped["smiles"] = ""\n    rows = list(grouped.reset_index()[["id", "smiles", "mass_sum", "n", "lo", "hi"]].itertuples(index=False, name=None))\n    db.executemany(f"INSERT INTO {table} (id, smiles, mass_sum, n, lo, hi) VALUES (?,?,?,?,?,?) "\n                   f"ON CONFLICT(id) DO NOTHING", rows)\n    db.commit()\n    resource_guard(cfg)\n    state_set(db, f"metadata_{kind}", True)\n\n\ndef ingest_splits(db, path, cfg):\n    if state_get(db, "splits_done"):\n        return\n    db.execute("DELETE FROM splits")\n    file = pq.ParquetFile(path)\n    has_ik14 = "ik14" in file.schema_arrow.names\n    adapter = SchemaAdapter(file.schema_arrow.names, "splits", ("molecule_id", "split3"))\n    columns = [adapter.mapping[k] for k in ("molecule_id", "split3")]\n    if has_ik14 and "ik14" not in columns:\n        columns.append("ik14")\n    for batch in file.iter_batches(cfg.BATCH_ROWS, columns=columns):\n        frame = adapter.adapt(batch)\n        if frame.isna().any().any() or not frame.split3.isin(["train", "val_model", "val_rank"]).all():\n            raise ValueError("splits must contain non-null, unique molecule_id and valid split3")\n        ik14_vals = frame["ik14"].astype(str) if "ik14" in frame else [None] * len(frame)\n        try:\n            db.executemany("INSERT INTO splits VALUES (?,?,?)",\n                           zip(frame.molecule_id.astype(str), frame.split3.astype(str), ik14_vals))\n        except sqlite3.IntegrityError as exc:\n            raise ValueError("splits has duplicate molecule_id assignments") from exc\n    mismatch = db.execute("SELECT COUNT(*) FROM (SELECT id FROM molecules EXCEPT SELECT id FROM splits)").fetchone()[0]\n    extra = db.execute("SELECT COUNT(*) FROM (SELECT id FROM splits EXCEPT SELECT id FROM molecules)").fetchone()[0]\n    if mismatch or extra:\n        raise ValueError(f"splits/train ID mismatch: missing={mismatch}, extra={extra}")\n    db.execute("UPDATE molecules SET split3=(SELECT split3 FROM splits WHERE splits.id=molecules.id), "\n               "ik14=(SELECT ik14 FROM splits WHERE splits.id=molecules.id)")\n    db.execute("CREATE INDEX IF NOT EXISTS mol_smiles ON molecules(smiles)")\n    db.execute("CREATE INDEX IF NOT EXISTS mol_split ON molecules(split3)")\n    state_set(db, "splits_done", True)\n\n\nUPSERT = """INSERT INTO structures VALUES (?,?,?,?,?,?,?)\nON CONFLICT(fast_key) DO UPDATE SET\n source_mask=structures.source_mask|excluded.source_mask,\n ik14=coalesce(structures.ik14,excluded.ik14)"""\n\n\ndef prepare_train(db, pool, cfg):\n    if state_get(db, "train_done"):\n        return state_get(db, "allowed_elements")\n    total = db.execute("SELECT COUNT(DISTINCT smiles) FROM molecules").fetchone()[0]\n    cursor = db.execute("SELECT smiles, max(ik14) FROM molecules GROUP BY smiles ORDER BY smiles")\n    allowed = BASE_ELEMENTS | EXTRA_ELEMENTS\n    tasks = ((r[0], TRAIN_BIT, tuple(sorted(allowed)), r[1]) for r in cursor)\n    statuses = Counter()\n    found_elements = set()\n    for result in pool_map(pool, standardize_chunk, tasks, cfg, total, "Train chemistry"):\n        good = []\n        mapping = []\n        for raw, status, row in result:\n            statuses[status] += 1\n            if row:\n                key, smiles, formula, mass, source, ik14, elements = row\n                good.append(row)\n                found_elements.update(elements.split(","))\n                mapping.append((raw, key, ik14, smiles, formula, mass, status))\n            else:\n                mapping.append((raw, None, None, None, None, None, status))\n        if not cfg.EXCLUDE_TRAIN_FROM_DB:\n            db.executemany(UPSERT, good)\n        db.executemany("INSERT OR REPLACE INTO train_structures VALUES (?,?,?,?,?,?,?)", mapping)\n        db.commit()\n    allowed = sorted(BASE_ELEMENTS | EXTRA_ELEMENTS)\n    state_set(db, "allowed_elements", allowed)\n    state_set(db, "train_rejections", dict(statuses))\n    state_set(db, "train_done", True)\n    db.execute("CREATE INDEX IF NOT EXISTS train_fast ON train_structures(fast_key)")\n    print("Train chemistry:", dict(statuses), "allowed elements:", allowed)\n    leakage = db.execute("SELECT COUNT(*) FROM (SELECT t.fast_key FROM molecules m JOIN train_structures t "\n                         "ON m.smiles=t.raw WHERE t.fast_key IS NOT NULL GROUP BY t.fast_key "\n                         "HAVING COUNT(DISTINCT m.split3)>1)").fetchone()[0]\n    if leakage:\n        raise ValueError(f"{leakage} standardized fast_key identities cross split3; regenerate splits")\n    return allowed\n\n\ndef smiles_column(columns, sample):\n    names = {str(c).lower(): c for c in columns}\n    preferred = SchemaAdapter.aliases["smiles"] + ["compound_smiles"]\n    candidates = [names[s] for s in preferred if s in names]\n    candidates += [c for c in columns if "smile" in str(c).lower() and c not in candidates]\n    candidates += [c for c in columns if c not in candidates]\n    for col in candidates:\n        values = [v for v in sample[col].tolist() if isinstance(v, str) and v.strip()][:32]\n        if not values:\n            continue\n        fraction = sum(Chem.MolFromSmiles(v) is not None for v in values) / len(values)\n        if fraction >= (0.1 if "smile" in str(col).lower() else 0.8):\n            return col\n    return None\n\n\ndef detect_delimiter(text, fmt):\n    if fmt == "tsv":\n        return "\\t"\n    if fmt == "csv":\n        return ","\n    if fmt == "smi":\n        return "\\t" if "\\t" in text[:1000] else " "\n    try:\n        return csv.Sniffer().sniff(text, delimiters="\\t, ;|").delimiter\n    except csv.Error:\n        return "\\t" if "\\t" in text else "," if "," in text else " "\n\n\ndef probe_zip_member(zf, member, m_fmt, m_size, path_str):\n    if m_fmt == "sdf":\n        return {"path": path_str, "zip_member": member, "format": "sdf",\n                "is_gz": False, "is_zip": True, "bytes": m_size,\n                "column": "<SDF mol block>", "rows_estimate": None}\n    with zf.open(member, "r") as raw:\n        with io.TextIOWrapper(raw, encoding="utf-8-sig", errors="replace", newline="") as handle:\n            lines = list(itertools.islice(handle, 129))\n    if not lines:\n        return None\n    text = "".join(lines)\n    delimiter = detect_delimiter(text, m_fmt)\n    records = list(csv.reader(lines, delimiter=delimiter, skipinitialspace=True))\n    records = [r for r in records if r]\n    if not records:\n        return None\n    header = any("smile" in str(x).lower() for x in records[0])\n    names = records[0] if header else list(range(len(records[0])))\n    width = len(names)\n    sample = pd.DataFrame([r[:width] + [None] * max(0, width - len(r))\n                           for r in records[int(header):]], columns=names)\n    col = smiles_column(names, sample)\n    if col is None:\n        return None\n    mean_bytes = len(text.encode("utf-8")) / max(1, len(lines))\n    return {"path": path_str, "zip_member": member, "format": m_fmt,\n            "is_gz": False, "is_zip": True, "bytes": m_size,\n            "column": col, "column_index": names.index(col), "header": header,\n            "delimiter": delimiter,\n            "rows_estimate": max(1, round(m_size / max(1.0, mean_bytes)))}\n\n\ndef probe_zip_source(path):\n    p = Path(path)\n    try:\n        with zipfile.ZipFile(p, "r") as zf:\n            candidates = [\n                (info.filename, Path(info.filename).suffix.lower()[1:], info.file_size)\n                for info in zf.infolist()\n                if not info.is_dir() and "__MACOSX" not in info.filename\n                and Path(info.filename).suffix.lower() in {".sdf", ".tsv", ".csv", ".smi", ".txt"}\n            ]\n            if not candidates:\n                return None\n            specs = [probe_zip_member(zf, m, fmt, sz, str(p)) for m, fmt, sz in candidates]\n            valid = [s for s in specs if s is not None]\n            return valid if valid else None\n    except Exception as exc:\n        print(f"[probe_zip_source] cannot inspect zip {path}: {exc}")\n        return None\n\n\ndef probe_delimited_source(p, format_name, is_gz, spec):\n    opener = (lambda: gzip.open(p, mode="rt", encoding="utf-8-sig", errors="replace", newline="")) if is_gz \\\n             else (lambda: p.open(encoding="utf-8-sig", errors="replace", newline=""))\n    try:\n        with opener() as handle:\n            lines = list(itertools.islice(handle, 129))\n    except Exception as exc:\n        print(f"[probe_delimited_source] cannot open {p}: {exc}")\n        return None\n    if not lines:\n        return None\n    text = "".join(lines)\n    delimiter = detect_delimiter(text, format_name)\n    records = list(csv.reader(lines, delimiter=delimiter, skipinitialspace=True))\n    records = [r for r in records if r]\n    if not records:\n        return None\n    header = any("smile" in str(x).lower() for x in records[0])\n    names = records[0] if header else list(range(len(records[0])))\n    width = len(names)\n    sample = pd.DataFrame([r[:width] + [None] * max(0, width - len(r))\n                           for r in records[int(header):]], columns=names)\n    col = smiles_column(names, sample)\n    if col is None:\n        return None\n    mean_bytes = len(text.encode("utf-8")) / max(1, len(lines))\n    approx_size = spec["bytes"] * 4 if is_gz else spec["bytes"]\n    spec.update(column=col, column_index=names.index(col), header=header,\n                delimiter=delimiter, rows_estimate=max(1, round(approx_size / max(1.0, mean_bytes))))\n    return spec\n\n\ndef probe_source(path):\n    p = Path(path)\n    name_lower = p.name.lower()\n    is_gz = name_lower.endswith(".gz")\n    if name_lower.endswith(".zip"):\n        return probe_zip_source(p)\n    base_ext = Path(p.stem).suffix.lower() if is_gz else p.suffix.lower()\n    format_name = base_ext[1:] if base_ext else "txt"\n    spec = {"path": str(p), "format": format_name, "is_gz": is_gz, "is_zip": False,\n            "bytes": p.stat().st_size}\n    if format_name == "parquet":\n        file = pq.ParquetFile(p)\n        columns = [f.name for f in file.schema_arrow if pa.types.is_string(f.type)\n                   or pa.types.is_large_string(f.type)]\n        if not columns or not file.metadata.num_rows:\n            return None\n        sample = next(file.iter_batches(batch_size=128, columns=columns)).to_pandas()\n        col = smiles_column(columns, sample)\n        if col is not None:\n            spec.update(column=col, rows_estimate=file.metadata.num_rows)\n            return spec\n        return None\n    if format_name == "sdf":\n        spec.update(column="<SDF mol block>", rows_estimate=None)\n        return spec\n    return probe_delimited_source(p, format_name, is_gz, spec)\n\n\ndef discover_sources(root, core_paths):\n    valid_extensions = {".smi", ".csv", ".tsv", ".sdf", ".parquet", ".txt",\n                        ".smi.gz", ".csv.gz", ".tsv.gz", ".sdf.gz", ".txt.gz", ".gz", ".zip"}\n    excluded = {Path(p).resolve() for p in core_paths}\n    core_dirs = {p.parent for p in excluded}\n    reserved = {"train.parquet", "test.parquet", "train_clean.parquet", "test_clean.parquet",\n                "splits.parquet", "sample_submission.csv", "db.parquet"}\n    sources = []\n    for path in sorted(Path(root).rglob("*")):\n        if not path.is_file():\n            continue\n        p_name = path.name.lower()\n        if not any(p_name.endswith(ext) for ext in valid_extensions):\n            continue\n        if path.resolve() in excluded or path.name in reserved or path.parent.resolve() in core_dirs:\n            print(f"[discovery] core input/output excluded: {path}")\n            continue\n        try:\n            probed = probe_source(path)\n        except (OSError, ValueError, pa.ArrowException, UnicodeError, csv.Error) as exc:\n            print(f"[discovery] cannot inspect {path}: {type(exc).__name__}: {exc}")\n            continue\n        if probed is None:\n            print(f"[discovery] no SMILES-like column: {path}")\n            continue\n        specs = probed if isinstance(probed, list) else [probed]\n        for spec in specs:\n            try:\n                parts = path.relative_to(root).parts\n                # Handle Kaggle\'s nested \'/kaggle/input/datasets/user/dataset_slug/\' paths\n                if parts[0] == "datasets" and len(parts) > 2:\n                    spec["dataset"] = parts[2]\n                else:\n                    spec["dataset"] = parts[0]\n            except ValueError:\n                spec["dataset"] = path.stem\n            if spec.get("is_zip") and spec.get("zip_member"):\n                spec["dataset"] += f"_{Path(spec[\'zip_member\']).stem}"\n            sources.append(spec)\n            desc = f"{spec.get(\'zip_member\', \'\')} {spec[\'column\']!r}"\n            print(f"[discovery] FOUND {path}: {desc.strip()}, estimated rows={spec[\'rows_estimate\']}")\n    datasets = sorted({s["dataset"] for s in sources})\n    if len(datasets) > 62:\n        raise ValueError("uint64 provenance supports train + 62 external datasets; combine attachments")\n    bits = {name: 1 << (i + 1) for i, name in enumerate(datasets)}\n    for spec in sources:\n        spec["source_mask"] = bits[spec["dataset"]]\n    if not sources:\n        print("No external structure databases found. Building DB from train SMILES only.")\n    print("Source flags:", {"train": TRAIN_BIT, **bits})\n    return sources\n\n\ndef iter_sdf_source(path, is_gz, is_zip, zip_member):\n    if is_zip:\n        with zipfile.ZipFile(path, "r") as zf:\n            with zf.open(zip_member, "r") as handle:\n                for mol in Chem.ForwardSDMolSupplier(handle, removeHs=True):\n                    yield Chem.MolToSmiles(mol) if mol is not None else None\n    elif is_gz:\n        with gzip.open(path, "rb") as handle:\n            for mol in Chem.ForwardSDMolSupplier(handle, removeHs=True):\n                yield Chem.MolToSmiles(mol) if mol is not None else None\n    else:\n        with path.open("rb") as handle:\n            for mol in Chem.ForwardSDMolSupplier(handle, removeHs=True):\n                yield Chem.MolToSmiles(mol) if mol is not None else None\n\n\ndef iter_delimited_source(path, spec):\n    is_gz, is_zip = spec.get("is_gz", False), spec.get("is_zip", False)\n    col = spec["column_index"]\n\n    def _read(reader):\n        if spec["header"]:\n            next(reader, None)\n        while True:\n            try:\n                row = next(reader)\n            except StopIteration:\n                break\n            except csv.Error as exc:\n                print(f"[delimited_source] warning: {exc}")\n                continue\n            yield row[col] if len(row) > col else None\n\n    if is_zip:\n        with zipfile.ZipFile(path, "r") as zf:\n            with zf.open(spec["zip_member"], "r") as raw:\n                with io.TextIOWrapper(raw, encoding="utf-8-sig", errors="replace", newline="") as handle:\n                    yield from _read(csv.reader(handle, delimiter=spec["delimiter"], skipinitialspace=True))\n    elif is_gz:\n        with gzip.open(path, mode="rt", encoding="utf-8-sig", errors="replace", newline="") as handle:\n            yield from _read(csv.reader(handle, delimiter=spec["delimiter"], skipinitialspace=True))\n    else:\n        with path.open(encoding="utf-8-sig", errors="replace", newline="") as handle:\n            yield from _read(csv.reader(handle, delimiter=spec["delimiter"], skipinitialspace=True))\n\n\ndef iter_source(spec, batch_rows=8192):\n    path = Path(spec["path"])\n    fmt = spec["format"]\n    if fmt == "parquet":\n        for batch in pq.ParquetFile(path).iter_batches(batch_rows, columns=[spec["column"]]):\n            yield from batch.column(0).to_pylist()\n    elif fmt == "sdf":\n        yield from iter_sdf_source(path, spec.get("is_gz", False), spec.get("is_zip", False), spec.get("zip_member"))\n    else:\n        yield from iter_delimited_source(path, spec)\n\n\ndef benchmark_standardization(sources, db, pool, allowed, cfg):\n    target = min(500, cfg.BENCHMARK_ROWS) if cfg.SMOKE_TEST else cfg.BENCHMARK_ROWS\n    source_iter = itertools.chain.from_iterable(iter_source(s) for s in sources)\n    if not sources:\n        source_iter = (r[0] for r in db.execute("SELECT DISTINCT smiles FROM molecules"))\n    sample = list(itertools.islice(source_iter, target))\n    tasks = ((s, 0, tuple(allowed), None) for s in sample)\n    start = time.perf_counter()\n    accepted = 0\n    for result in pool_map(pool, standardize_chunk, tasks, cfg, len(sample), "20k chemistry benchmark"):\n        accepted += sum(row is not None for _, _, row in result)\n    seconds = time.perf_counter() - start\n    rate = len(sample) / max(seconds, 1e-9)\n    estimates = [s["rows_estimate"] for s in sources]\n    estimated_rows = sum(n for n in estimates if n is not None) if sources else len(sample)\n    report = {"sample_rows": len(sample), "accepted": accepted, "rows_per_second": rate,\n              "estimated_rows": estimated_rows, "unknown_size_sources": estimates.count(None),\n              "projected_seconds_known_sources": estimated_rows / max(rate, 1e-9)}\n    print("Standardization projection:", report)\n    return report\n\n\ndef ingest_external(db, sources, pool, allowed, cfg):\n    all_stats = state_get(db, "external_rejections", {})\n    for source in sources:\n        src_key = source["path"] + (f"::{source[\'zip_member\']}" if source.get("zip_member") else "")\n        if db.execute("SELECT done FROM sources WHERE path=?", (src_key,)).fetchone() == (1,):\n            print("Resume: completed external source", src_key)\n            continue\n        counts = Counter()\n        tasks = ((s, source["source_mask"], tuple(allowed), None)\n                 for s in iter_source(source, cfg.BATCH_ROWS))\n        desc_name = f"{Path(source[\'path\']).name}:{source[\'zip_member\']}" if source.get("zip_member") else Path(source["path"]).name\n        for result in pool_map(pool, standardize_chunk, tasks, cfg,\n                               source["rows_estimate"], desc_name):\n            rows = []\n            for _, status, row in result:\n                counts[status] += 1\n                if row:\n                    rows.append(row)\n            db.executemany(UPSERT, rows)\n            db.commit()\n        db.execute("INSERT OR REPLACE INTO sources VALUES (?,1)", (src_key,))\n        all_stats[src_key] = dict(counts)\n        state_set(db, "external_rejections", all_stats)\n        print("Ingested:", src_key, dict(counts))\n    db.commit()\n    return all_stats\n\n\ndef apply_mass_cap(db, cfg):\n    before = db.execute("SELECT COUNT(*) FROM structures").fetchone()[0]\n    bounds = db.execute("SELECT min(mass_sum/n), max(mass_sum/n) FROM molecules WHERE mass_sum IS NOT NULL").fetchone()\n    report = {"before": before, "applied": before > cfg.MAX_DB_ROWS}\n    if before > cfg.MAX_DB_ROWS:\n        if bounds[0] is None:\n            raise ValueError("Cannot apply mass cap without valid train neutral masses")\n        low, high = max(0.0, float(bounds[0]) - 100.0), float(bounds[1]) + 100.0\n        db.execute("DELETE FROM structures WHERE mass < ? OR mass > ?", (low, high))\n        db.commit()\n        report.update(low=low, high=high, bound_source="train")\n        print(f"DB > {cfg.MAX_DB_ROWS:,}: retaining train mass window [{low:.6f}, {high:.6f}] Da")\n    report["after"] = db.execute("SELECT COUNT(*) FROM structures").fetchone()[0]\n    print("Mass-cap report:", report)\n    state_set(db, "mass_cap", report)\n    return report\n\n\nclass BitOr:\n    def __init__(self):\n        self.value = 0\n\n    def step(self, value):\n        self.value |= int(value)\n\n    def finalize(self):\n        return self.value\n\n\ndef build_identity_groups(db):\n    db.create_aggregate("bit_or", 1, BitOr)\n    db.executescript("""\n    DROP TABLE IF EXISTS grouped;\n    CREATE TABLE grouped AS\n    SELECT fast_key AS identity,\n           fast_key AS representative,\n           bit_or(source_mask) AS source_mask,\n           max(ik14) AS ik14,\n           NULL AS row_id\n    FROM structures GROUP BY fast_key;\n    CREATE UNIQUE INDEX grouped_identity ON grouped(identity);\n    CREATE UNIQUE INDEX grouped_rep ON grouped(representative);\n    """)\n    db.commit()\n    n = db.execute("SELECT COUNT(*) FROM grouped").fetchone()[0]\n    print(f"Candidate pool bulk de-duplication: {n:,} rows uniformly indexed by fast_key")\n    return n\n\n\ndef memmap_new(path, dtype, shape):\n    return np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n\n\ndef map_read(path):\n    return np.load(path, mmap_mode="r", allow_pickle=False)\n\n\ndef build_mass_files(db, cfg, n):\n    root = Path(cfg.output_dir)\n    widths, fp_bytes = fp_layout()\n    needed = n * (fp_bytes + 8 + 8 + 1 + 8)\n    if shutil.disk_usage(root).free < needed + cfg.MIN_FREE_DISK_GB * 1e9:\n        raise OSError(f"Need at least {needed/1e9:.2f} GB for arrays plus parquet/index; insufficient disk")\n    mass = memmap_new(root / "mass.npy", np.float64, (n,))\n    order = memmap_new(root / "mass_sorted_idx.npy", np.int64, (n,))\n    schema = pa.schema([("row_id", pa.int64()), ("ik14", pa.string()), ("smiles", pa.string()),\n                        ("formula", pa.string()), ("mass", pa.float64()), ("source_mask", pa.uint64()),\n                        ("fast_key", pa.string()), ("identity_kind", pa.string())])\n    sql = ("SELECT g.ik14,s.smiles,s.formula,s.mass,g.source_mask,s.fast_key,g.identity "\n           "FROM grouped g JOIN structures s ON s.fast_key=g.representative "\n           "ORDER BY s.mass,s.fast_key")\n    cursor = db.execute(sql)\n    start = 0\n    offsets = [0]\n    with pq.ParquetWriter(root / "db.parquet", schema, compression="zstd") as writer:\n        for rows in tqdm(batched(cursor, cfg.BATCH_ROWS), total=math.ceil(n/cfg.BATCH_ROWS), desc="Mass-sorted export"):\n            stop = start + len(rows)\n            ids = np.arange(start, stop, dtype=np.int64)\n            mass[start:stop] = [r[3] for r in rows]\n            order[start:stop] = ids\n            values = [ids, [r[0] for r in rows], [r[1] for r in rows], [r[2] for r in rows],\n                      mass[start:stop], [r[4] for r in rows], [r[5] for r in rows],\n                      ["canonical" if r[0] else "fast_only" for r in rows]]\n            writer.write_table(pa.Table.from_arrays([pa.array(v, type=f.type)\n                                                     for v, f in zip(values, schema)], schema=schema))\n            db.executemany("UPDATE grouped SET row_id=? WHERE identity=?",\n                           [(int(i), r[6]) for i, r in zip(ids, rows)])\n            db.commit()\n            start = stop\n            offsets.append(stop)\n            mass.flush()\n            order.flush()\n            resource_guard(cfg)\n    np.save(root / "row_group_offsets.npy", np.array(offsets, dtype=np.int64))\n    del mass, order\n    gc.collect()\n    return widths, fp_bytes\n\n\ndef build_disk_indexes(db, cfg, n):\n    root = Path(cfg.output_dir)\n    index_path = root / "index.sqlite"\n    index = open_sql(index_path)\n    index.executescript("""\n    DROP TABLE IF EXISTS aliases; DROP TABLE IF EXISTS formula_ranges;\n    CREATE TABLE aliases(fast_key TEXT PRIMARY KEY, row_id INTEGER, source_mask INTEGER);\n    CREATE INDEX aliases_row ON aliases(row_id);\n    CREATE TABLE formula_ranges(formula TEXT PRIMARY KEY, start INTEGER, stop INTEGER);\n    """)\n    cursor = db.execute("SELECT s.fast_key,g.row_id,s.source_mask FROM structures s "\n                        "JOIN grouped g ON g.identity=s.fast_key")\n    for rows in batched(cursor, cfg.BATCH_ROWS):\n        index.executemany("INSERT INTO aliases VALUES (?,?,?)", rows)\n    formula_ids = memmap_new(root / "formula_row_ids.npy", np.int64, (n,))\n    cursor = db.execute("SELECT s.formula,g.row_id FROM grouped g JOIN structures s "\n                        "ON s.fast_key=g.representative ORDER BY s.formula,g.row_id")\n    pos, start, previous = 0, 0, None\n    offsets = [0]\n    for rows in tqdm(batched(cursor, cfg.BATCH_ROWS), desc="Formula CSR index"):\n        formula_ids[pos:pos+len(rows)] = [r[1] for r in rows]\n        for formula, group in itertools.groupby(rows, key=lambda r: r[0]):\n            count = sum(1 for _ in group)\n            if previous is not None and formula != previous:\n                index.execute("INSERT INTO formula_ranges VALUES (?,?,?)", (previous, start, pos))\n                offsets.append(pos)\n                start = pos\n            previous = formula\n            pos += count\n        formula_ids.flush()\n        resource_guard(cfg)\n    if previous is not None:\n        index.execute("INSERT INTO formula_ranges VALUES (?,?,?)", (previous, start, pos))\n        offsets.append(pos)\n    np.save(root / "formula_offsets.npy", np.array(offsets, dtype=np.int64))\n    index.commit()\n    index.close()\n    del formula_ids\n    gc.collect()\n\n\ndef query_masses(db):\n    return (r[0] for r in db.execute("SELECT mass_sum/n FROM molecules WHERE mass_sum IS NOT NULL "\n                                    "UNION ALL SELECT mass_sum/n FROM tests WHERE mass_sum IS NOT NULL"))\n\n\ndef mark_fp_rows(db, cfg, n):\n    root = Path(cfg.output_dir)\n    needed = memmap_new(root / "fp_needed.npy", np.uint8, (n,))\n    mass = map_read(root / "mass.npy")\n    for values in batched(query_masses(db), cfg.BATCH_ROWS):\n        q = np.array(values, dtype=np.float64)\n        delta = q * cfg.FP_WINDOW_PPM * 1e-6\n        left = np.searchsorted(mass, q - delta)\n        right = np.searchsorted(mass, q + delta, side="right")\n        for lo, hi in zip(left, right):\n            needed[lo:hi] = 1\n        needed.flush()\n    cursor = db.execute("SELECT DISTINCT g.row_id FROM grouped g JOIN structures s "\n                        "ON g.identity=s.fast_key "\n                        "WHERE EXISTS(SELECT 1 FROM train_structures t WHERE t.fast_key=s.fast_key)")\n    for rows in batched(cursor, cfg.BATCH_ROWS):\n        needed[np.array([r[0] for r in rows], dtype=np.int64)] = 1\n    needed.flush()\n    count = sum(int(np.count_nonzero(needed[i:i+cfg.BATCH_ROWS])) for i in range(0, n, cfg.BATCH_ROWS))\n    del needed, mass\n    gc.collect()\n    print(f"FP selection: {count:,}/{n:,} query-window rows plus all retained train parents")\n    return count\n\n\ndef iter_fp_rows(root, selection=None, batch_rows=8192):\n    for batch in pq.ParquetFile(Path(root)/"db.parquet").iter_batches(\n            batch_rows, columns=["row_id", "smiles"]):\n        ids = batch.column(0).to_numpy()\n        if selection is not None:\n            keep = np.asarray(selection[ids], dtype=bool)\n            batch = batch.filter(pa.array(keep))\n            ids = ids[keep]\n        yield from zip(ids.tolist(), batch.column(1).to_pylist())\n\n\ndef benchmark_fingerprints(cfg, pool, n, selected):\n    target = min(500, cfg.BENCHMARK_ROWS) if cfg.SMOKE_TEST else cfg.BENCHMARK_ROWS\n    rng = np.random.default_rng(cfg.seed)\n    sample_ids = np.sort(rng.choice(n, min(n, target), replace=False)) if n else np.array([], dtype=int)\n    sample = []\n    file = pq.ParquetFile(Path(cfg.output_dir)/"db.parquet")\n    for batch in file.iter_batches(cfg.BATCH_ROWS, columns=["row_id", "smiles"]):\n        ids = batch.column(0).to_numpy()\n        left = np.searchsorted(sample_ids, ids[0], side="left") if len(ids) else 0\n        right = np.searchsorted(sample_ids, ids[-1], side="right") if len(ids) else 0\n        if right > left:\n            take = sample_ids[left:right] - ids[0]\n            small = batch.take(pa.array(take))\n            sample.extend(zip(small.column(0).to_pylist(), small.column(1).to_pylist()))\n    start = time.perf_counter()\n    for _ in pool_map(pool, fingerprint_chunk, sample, cfg, len(sample), "20k FP benchmark"):\n        pass\n    seconds = time.perf_counter() - start\n    rate = len(sample) / max(seconds, 1e-9)\n    report = {"sample_rows": len(sample), "rows_per_second": rate,\n              "full_db_seconds": n/max(rate, 1e-9), "selected_seconds": selected/max(rate, 1e-9),\n              "FULL_DB_FP": cfg.FULL_DB_FP, "benchmark_fps_persisted": False}\n    print("FP projection:", report)\n    return report\n\n\ndef write_fingerprints(cfg, pool, n, width):\n    root = Path(cfg.output_dir)\n    valid_path, fp_path = root/"fp_valid.npy", root/"fp_packed.npy"\n    if valid_path.exists() and fp_path.exists():\n        fp = np.load(fp_path, mmap_mode="r+", allow_pickle=False)\n        valid = np.load(valid_path, mmap_mode="r+", allow_pickle=False)\n        if fp.shape != (n, width) or valid.shape != (n,):\n            raise ValueError("Incompatible fingerprint checkpoint; use a fresh output_dir")\n    else:\n        fp = memmap_new(fp_path, np.uint8, (n, width))\n        valid = memmap_new(valid_path, np.uint8, (n,))\n    needed = map_read(root/"fp_needed.npy")\n    def remaining():\n        for row_id, smiles in iter_fp_rows(root, None if cfg.FULL_DB_FP else needed, cfg.BATCH_ROWS):\n            if not valid[row_id]:\n                yield row_id, smiles\n    for result in pool_map(pool, fingerprint_chunk, remaining(), cfg, desc="Packed fingerprints"):\n        ids = np.array([r[0] for r in result], dtype=np.int64)\n        fp[ids] = np.stack([r[1] for r in result])\n        fp.flush()\n        valid[ids] = 1\n        valid.flush()\n    written = sum(int(np.count_nonzero(valid[i:i+cfg.BATCH_ROWS])) for i in range(0, n, cfg.BATCH_ROWS))\n    del fp, valid, needed\n    gc.collect()\n    print(f"Stored {written:,}/{n:,} fingerprints; missing entries computed lazily at query time")\n    return written\n\n\ndef configure(db_dir=None, exclusion_scope="train", cache_dir=None, row_group_cache=2,\n              ik14_cache_size=100000, fp_cache_size=32768, cache_commit_rows=5000):\n    global CTX\n    if exclusion_scope not in ("train", "all"):\n        raise ValueError("exclusion_scope must be \'train\' or \'all\'")\n    if CTX:\n        flush_cache()\n        CTX["index"].close()\n        CTX["cache"].close()\n    _MEM_IK14_CACHE.clear()\n    _MEM_FP_CACHE.clear()\n    root = Path(db_dir or WORK/"artifacts"/"candidates").resolve()\n    metadata = json.loads((root/"db_metadata.json").read_text())\n    if metadata["fp_spec"] != load_evalkit().FP_SPEC:\n        raise ValueError("FP_SPEC differs from database fingerprint schema")\n    cache_root = Path(cache_dir or WORK/"candidate_cache"/metadata["build_id"])\n    cache_root.mkdir(parents=True, exist_ok=True)\n    cache = open_sql(cache_root/"lazy.sqlite")\n    # Only the disposable inference cache changes durability; build DBs keep theirs.\n    cache.execute("PRAGMA journal_mode=WAL")\n    cache.execute("PRAGMA synchronous=NORMAL")\n    cache.execute("CREATE TABLE IF NOT EXISTS chemistry(row_id INTEGER PRIMARY KEY, ik14 TEXT, fp BLOB)")\n    cache.commit()\n    CTX = {"root": root, "meta": metadata, "mass": map_read(root/"mass.npy"),\n           "order": map_read(root/"mass_sorted_idx.npy"), "formula_ids": map_read(root/"formula_row_ids.npy"),\n           "offsets": map_read(root/"row_group_offsets.npy"), "fp": map_read(root/"fp_packed.npy"),\n           "valid": map_read(root/"fp_valid.npy"), "index": open_sql(root/"index.sqlite", True),\n           "parquet": pq.ParquetFile(root/"db.parquet"), "cache": cache,\n           "scope": exclusion_scope, "groups": OrderedDict(), "max_groups": max(1, row_group_cache),\n           "ik14_cache_size": max(1, ik14_cache_size), "fp_cache_size": max(1, fp_cache_size),\n           "cache_commit_rows": max(1, cache_commit_rows), "cache_pending": 0}\n    return metadata\n\n\ndef context():\n    if CTX is None:\n        configure()\n    return CTX\n\n\ndef flush_cache():\n    """Persist pending lazy chemistry at split boundaries or before reconfiguration."""\n    if CTX is not None and CTX["cache"].in_transaction:\n        CTX["cache"].commit()\n    if CTX is not None:\n        CTX["cache_pending"] = 0\n\n\ndef cache_written(count):\n    state = context()\n    state["cache_pending"] = state.get("cache_pending", 0) + count\n    if state["cache_pending"] >= state.get("cache_commit_rows", 5000):\n        flush_cache()\n\n\ndef memory_cache_put(cache, key, value, limit):\n    cache[key] = value\n    cache.move_to_end(key)\n    while len(cache) > limit:\n        cache.popitem(last=False)\n\n\ndef prefetch_canonical_keys(row_ids):\n    """One batched SELECT for uncached IDs, including negative lookup caching."""\n    state = context()\n    ids = list(dict.fromkeys(int(i) for i in row_ids if int(i) not in _MEM_IK14_CACHE))\n    for chunk in batched(ids, 500):\n        placeholders = \',\'.join(\'?\' for _ in chunk)\n        rows = dict(state["cache"].execute(\n            f"SELECT row_id,ik14 FROM chemistry WHERE row_id IN ({placeholders})", chunk))\n        for row_id in chunk:\n            memory_cache_put(_MEM_IK14_CACHE, row_id, rows.get(row_id),\n                             state.get("ik14_cache_size", 100000))\n\n\ndef fetch_rows(ids):\n    state = context()\n    ids = np.asarray(ids, dtype=np.int64)\n    if len(ids) == 0:\n        return pd.DataFrame(columns=DB_COLUMNS)\n    if (ids < 0).any() or (ids >= len(state["mass"])).any():\n        raise IndexError("candidate row_id outside database")\n    if len(ids) > state["meta"]["max_materialized_rows"]:\n        raise MemoryError("Too many rows to materialize; use iter_by_mass / iter_by_formula")\n    groups = np.searchsorted(state["offsets"], ids, side="right") - 1\n    chunks = []\n    for group in np.unique(groups):\n        cache = state["groups"]\n        if int(group) not in cache:\n            cache[int(group)] = state["parquet"].read_row_group(int(group), columns=DB_COLUMNS)\n        cache.move_to_end(int(group))\n        while len(cache) > state["max_groups"]:\n            cache.popitem(last=False)\n        positions = np.flatnonzero(groups == group)\n        offsets = ids[positions] - state["offsets"][group]\n        frame = cache[int(group)].take(pa.array(offsets)).to_pandas()\n        frame["_position"] = positions\n        chunks.append(frame)\n    return pd.concat(chunks).sort_values("_position").drop(columns="_position").reset_index(drop=True)\n\n\ndef excluded_rows(exclude_keys):\n    if not exclude_keys:\n        return set()\n    result = set()\n    for keys in batched(sorted(exclude_keys), 500):\n        placeholders = ",".join("?" for _ in keys)\n        rows = context()["index"].execute(\n            f"SELECT row_id FROM aliases WHERE fast_key IN ({placeholders})", keys)\n        result.update(r[0] for r in rows)\n    return result\n\n\ndef filter_sources(frame, exclude_keys=None, external_only=False, excluded_ids=None):\n    if frame.empty:\n        return frame\n    state = context()\n    rejected = excluded_rows(exclude_keys) if excluded_ids is None else excluded_ids\n    mask = frame.source_mask.to_numpy(dtype=np.uint64).copy()\n    selected = frame.row_id.isin(rejected).to_numpy()\n    if state["scope"] == "all":\n        mask[selected] = 0\n    else:\n        mask[selected] &= np.uint64(0xFFFFFFFFFFFFFFFE)\n    if external_only:\n        mask &= np.uint64(0xFFFFFFFFFFFFFFFE)\n    frame = frame.copy()\n    frame["source_mask"] = mask\n    return frame.loc[mask != 0].reset_index(drop=True)\n\n\ndef mass_bounds(neutral_mass, ppm):\n    if not np.isfinite(neutral_mass) or neutral_mass <= 0 or not np.isfinite(ppm) or ppm < 0:\n        raise ValueError("neutral_mass must be positive/finite; ppm nonnegative/finite")\n    delta = float(neutral_mass) * float(ppm) * 1e-6\n    mass = context()["mass"]\n    return int(np.searchsorted(mass, neutral_mass - delta)), int(np.searchsorted(mass, neutral_mass + delta, side="right"))\n\n\ndef iter_by_mass(neutral_mass, ppm, exclude_keys=None, batch_rows=8192):\n    lo, hi = mass_bounds(neutral_mass, ppm)\n    blocked = excluded_rows(exclude_keys)\n    for start in range(lo, hi, batch_rows):\n        ids = context()["order"][start:min(hi, start + batch_rows)]\n        yield filter_sources(fetch_rows(ids), excluded_ids=blocked)\n\n\ndef query_by_mass(neutral_mass, ppm, exclude_keys=None):\n    lo, hi = mass_bounds(neutral_mass, ppm)\n    return filter_sources(fetch_rows(context()["order"][lo:hi]), exclude_keys)\n\n\ndef hill_formula(formula):\n    if not isinstance(formula, str) or not formula:\n        raise ValueError("Expected a nonempty neutral molecular formula")\n    parts = re.findall(r"([A-Z][a-z]?)([0-9]*)", formula)\n    if "".join(a+b for a, b in parts) != formula:\n        raise ValueError("Formula must be an uncharged, non-isotopic elemental formula")\n    counts = Counter()\n    for symbol, number in parts:\n        count = int(number or 1)\n        if count <= 0 or symbol not in context()["meta"]["allowed_elements"]:\n            raise ValueError(f"Invalid formula element/count: {symbol}{number}")\n        counts[symbol] += count\n    keys = ([s for s in ("C", "H") if s in counts] + sorted(set(counts) - {"C", "H"})\n            if "C" in counts else sorted(counts))\n    return "".join(k + (str(counts[k]) if counts[k] != 1 else "") for k in keys)\n\n\ndef formula_span(formula):\n    row = context()["index"].execute("SELECT start,stop FROM formula_ranges WHERE formula=?",\n                                     (hill_formula(formula),)).fetchone()\n    return row if row else (0, 0)\n\n\ndef iter_by_formula(formula, exclude_keys=None, batch_rows=8192):\n    lo, hi = formula_span(formula)\n    blocked = excluded_rows(exclude_keys)\n    for start in range(lo, hi, batch_rows):\n        ids = context()["formula_ids"][start:min(hi, start + batch_rows)]\n        yield filter_sources(fetch_rows(ids), excluded_ids=blocked)\n\n\ndef query_by_formula(formula, exclude_keys=None):\n    lo, hi = formula_span(formula)\n    return filter_sources(fetch_rows(context()["formula_ids"][lo:hi]), exclude_keys)\n\n\ndef get_fingerprints(row_ids, smiles=None):\n    """Get packed bits, computing missing rows once in a batch using supplied SMILES."""\n    state = context()\n    ids = np.asarray(row_ids, dtype=np.int64)\n    if smiles is not None:\n        smiles = np.asarray(smiles, dtype=object)\n        if smiles.shape != ids.shape:\n            raise ValueError("smiles must align positionally with row_ids")\n    if len(ids) == 0:\n        return np.empty((0, state["meta"]["fp_bytes"]), dtype=np.uint8)\n    if len(ids) > state["meta"]["max_materialized_rows"]:\n        raise MemoryError("Request fingerprints in bounded batches")\n    if (ids < 0).any() or (ids >= len(state["mass"])).any():\n        raise IndexError("candidate row_id outside database")\n    out = np.empty((len(ids), state["meta"]["fp_bytes"]), dtype=np.uint8)\n    ready = np.asarray(state["valid"][ids], dtype=bool)\n    out[ready] = state["fp"][ids[ready]]\n    missing = np.flatnonzero(~ready)\n    if len(missing) > 0:\n        positions = {}\n        for position in missing:\n            positions.setdefault(int(ids[position]), []).append(int(position))\n        resolved, uncached = {}, []\n        for row_id in positions:\n            if row_id in _MEM_FP_CACHE:\n                resolved[row_id] = _MEM_FP_CACHE[row_id]\n                _MEM_FP_CACHE.move_to_end(row_id)\n            else:\n                uncached.append(row_id)\n        for chunk in batched(uncached, 500):\n            placeholders = \',\'.join(\'?\' for _ in chunk)\n            for row_id, blob in state["cache"].execute(\n                    f"SELECT row_id,fp FROM chemistry WHERE row_id IN ({placeholders})", chunk):\n                if blob is not None:\n                    resolved[int(row_id)] = np.frombuffer(blob, dtype=np.uint8)\n        to_compute = [row_id for row_id in uncached if row_id not in resolved]\n        if to_compute:\n            if smiles is None:\n                missing_smiles = fetch_rows(to_compute).smiles.to_numpy()\n            else:\n                missing_smiles = [smiles[positions[row_id][0]] for row_id in to_compute]\n            computed = fingerprint_chunk(list(zip(to_compute, missing_smiles)))\n            state["cache"].executemany(\n                "INSERT INTO chemistry(row_id,fp) VALUES (?,?) "\n                "ON CONFLICT(row_id) DO UPDATE SET fp=excluded.fp",\n                [(int(row_id), packed.tobytes()) for row_id, packed in computed])\n            resolved.update((int(row_id), packed) for row_id, packed in computed)\n            cache_written(len(computed))\n        for row_id, locations in positions.items():\n            packed = resolved[row_id]\n            out[locations] = packed\n            memory_cache_put(_MEM_FP_CACHE, row_id, packed, state.get("fp_cache_size", 32768))\n    return out\n\n\ndef canonical_key(row_id, smiles, known=None):\n    if isinstance(known, str) and known:\n        return known\n    state = context()\n    row_id = int(row_id)\n    if row_id not in _MEM_IK14_CACHE:\n        prefetch_canonical_keys([row_id])\n    key = _MEM_IK14_CACHE[row_id]\n    _MEM_IK14_CACHE.move_to_end(row_id)\n    if key:\n        return key\n    key = load_evalkit().canon_ik14(Chem.MolFromSmiles(smiles))\n    if not key:\n        raise ValueError(f"evalkit.canon_ik14 failed at row {row_id}")\n    state["cache"].execute("INSERT INTO chemistry(row_id,ik14) VALUES (?,?) "\n                           "ON CONFLICT(row_id) DO UPDATE SET ik14=excluded.ik14", (int(row_id), key))\n    memory_cache_put(_MEM_IK14_CACHE, row_id, key, state.get("ik14_cache_size", 100000))\n    cache_written(1)\n    return key\n\n\ndef finalize_candidates(ranked, top_n=256, limit=25, canonical_limit=None):\n    """Deduplicate a shortlist; optionally canonicalize only its leading unique hits.\n\n    canonical_limit=25 preserves canonical top-25 submission identities and uses\n    fast_key deduplication for the remaining Stage 09 rows. Their canon_ik14 is\n    null, explicitly distinguishing unevaluated tautomers from canonical hits.\n    The default retains the full-canonical behavior for existing Stage 04 callers.\n    """\n    if top_n < 1 or limit < 1:\n        raise ValueError("top_n and limit must be positive")\n    shortlist = ranked.head(top_n).copy()\n    if canonical_limit is not None and canonical_limit < 1:\n        raise ValueError("canonical_limit must be positive")\n    merged = OrderedDict()\n    fast_seen = set()\n    canonical_count = 0\n    # Batching negative lookups removes per-row SQL, while bounded RAM caches\n    # absorb recurring row IDs from overlapping windows.\n    unknown = shortlist.loc[~shortlist.ik14.map(lambda k: isinstance(k, str) and bool(k)), \'row_id\']\n    prefetch_canonical_keys(unknown)\n    for row in shortlist.to_dict("records"):\n        if canonical_limit is not None and canonical_count >= canonical_limit:\n            fast = row[\'fast_key\']\n            if fast in fast_seen:\n                continue\n            key = (\'fast\', fast)\n            row.update(ik14=fast, canon_ik14=None, identity_kind="fast_only")\n        else:\n            canonical = canonical_key(row["row_id"], row["smiles"], row.get("ik14"))\n            key = (\'canonical\', canonical)\n            row.update(ik14=canonical, identity_kind="canonical")\n            if canonical_limit is not None:\n                row[\'canon_ik14\'] = canonical\n            if key not in merged:\n                canonical_count += 1\n        if key not in merged:\n            row.update(member_row_ids=[int(row["row_id"])])\n            merged[key] = row\n        else:\n            merged[key]["source_mask"] |= int(row["source_mask"])\n            merged[key]["member_row_ids"].append(int(row["row_id"]))\n        fast_seen.add(row.get(\'fast_key\'))\n        if canonical_limit is not None and len(merged) >= limit:\n            break\n    columns = list(shortlist.columns) + ["member_row_ids"]\n    if canonical_limit is not None and \'canon_ik14\' not in columns:\n        columns.append(\'canon_ik14\')\n    records = list(merged.values())[:limit]\n    result = pd.DataFrame(records, columns=columns)\n    # Construct directly from Python integers: pandas inference can otherwise\n    # round mixed high-bit masks through float64 before a later uint64 cast.\n    for column in (\'source_mask\', \'scoring_source_mask\'):\n        if column in result:\n            result[column] = np.array([int(r[column]) for r in records], dtype=np.uint64)\n    return result\n\n\ndef external_coverage(db):\n    denominator = db.execute("SELECT COUNT(*) FROM molecules").fetchone()[0]\n    valid = db.execute("SELECT COUNT(*) FROM molecules m JOIN train_structures t ON m.smiles=t.raw "\n                       "WHERE t.fast_key IS NOT NULL").fetchone()[0]\n    fast = db.execute("SELECT COUNT(*) FROM molecules m JOIN train_structures t ON m.smiles=t.raw "\n                      "WHERE EXISTS(SELECT 1 FROM structures s WHERE s.fast_key=t.fast_key "\n                      "AND (s.source_mask & ~1)!=0)").fetchone()[0]\n    result = {"train_molecules": denominator, "valid_train_molecules": valid,\n              "external_fast_key_hits": fast,\n              "external_coverage_percent": 100 * fast / max(denominator, 1),\n              "canonical_coverage_is_lower_bound": True,\n              "denominator_includes_rejected_train": True}\n    print("External DB coverage:", result)\n    return result\n\n\ndef validation_exclusions(db):\n    return {r[0] for r in db.execute("SELECT DISTINCT t.fast_key FROM molecules m "\n                                     "JOIN train_structures t ON m.smiles=t.raw "\n                                     "WHERE m.split3!=\'train\' AND t.fast_key IS NOT NULL")}\n\n\ndef scan_oracle_query(mass, ppm, truth, true_row_ids, blocked, external_only, top_n=256):\n    count, present, unknown, shortlist = 0, False, 0, []\n    for frame in iter_by_mass(mass, ppm):\n        frame = filter_sources(frame, external_only=external_only, excluded_ids=blocked)\n        count += len(frame)\n        if truth:\n            present |= bool(frame.row_id.isin(true_row_ids).any() or frame.ik14.eq(truth).any())\n        pending = frame.loc[frame.ik14.isna()]\n        unknown += len(pending)\n        if truth and not present:\n            errors = np.abs(pending.mass.to_numpy(dtype=float) - mass)\n            for offset in np.argsort(errors, kind="stable")[:top_n]:\n                row = pending.iloc[offset]\n                item = (-float(errors[offset]), -int(row.row_id), str(row.smiles))\n                if len(shortlist) < top_n:\n                    heapq.heappush(shortlist, item)\n                elif item > shortlist[0]:\n                    heapq.heapreplace(shortlist, item)\n    checked = 0\n    if truth and not present:\n        for _, negative_id, smiles in sorted(shortlist, reverse=True):\n            checked += 1\n            if canonical_key(-negative_id, smiles) == truth:\n                present = True\n                break\n    unresolved = bool(truth and not present and unknown > checked)\n    context()["cache"].commit()\n    return count, int(present), int(present or unresolved), int(unresolved)\n\n\ndef report_coverage(db, cfg):\n    root = Path(cfg.output_dir)\n    blocked = excluded_rows(validation_exclusions(db))\n    modes = [("all_sources_unexcluded", "train", False, set()),\n             ("heldout_removed_train", "train", False, blocked),\n             ("heldout_removed_all", "all", False, blocked),\n             ("external_only", "train", True, set())]\n    rows = []\n    query_sql = ("SELECT m.id,m.split3,m.mass_sum/m.n,m.ik14,t.fast_key FROM molecules m "\n                 "LEFT JOIN train_structures t ON t.raw=m.smiles WHERE m.split3!=\'train\' "\n                 "UNION ALL SELECT id,\'test\',mass_sum/n,NULL,NULL FROM tests")\n    total = db.execute("SELECT COUNT(*) FROM molecules WHERE split3!=\'train\'").fetchone()[0]\n    total += db.execute("SELECT COUNT(*) FROM tests").fetchone()[0]\n    schema = pa.schema([("molecule_id", pa.string()), ("split", pa.string()), ("configuration", pa.string()),\n                        ("ppm", pa.int32()), ("candidates", pa.int64()), ("oracle_lower", pa.float64()),\n                        ("oracle_upper", pa.float64()), ("unresolved", pa.int64())])\n    with pq.ParquetWriter(root/"query_coverage.parquet", schema, compression="zstd") as writer:\n        for mol_id, split, mass, truth, key in tqdm(db.execute(query_sql), total=total, desc="Molecule-level oracle audit"):\n            true_ids = excluded_rows({key}) if key else set()\n            for name, scope, external, excluded in modes:\n                context()["scope"] = scope\n                for ppm in cfg.REPORT_PPM:\n                    count, lower, upper, unresolved = scan_oracle_query(\n                        mass, ppm, truth, true_ids, excluded, external, cfg.ORACLE_CANON_TOP_N)\n                    rows.append((mol_id, split, name, ppm, count, lower if split != "test" else None,\n                                 upper if split != "test" else None, unresolved))\n            if len(rows) >= cfg.BATCH_ROWS:\n                writer.write_table(pa.Table.from_pylist([dict(zip(schema.names, r)) for r in rows], schema=schema))\n                rows.clear()\n                resource_guard(cfg)\n        if rows:\n            writer.write_table(pa.Table.from_pylist([dict(zip(schema.names, r)) for r in rows], schema=schema))\n    context()["scope"] = cfg.EXCLUSION_SCOPE\n    summaries = aggregate_query_reports(root/"query_coverage.parquet", db, cfg)\n    return {"external": external_coverage(db), "queries": summaries,\n            "identity_note": "fast-key pool sizes; top-256 tautomer audit gives oracle interval",\n            "leaky_diagnostic": "all_sources_unexcluded includes held-out train-file structures",\n            "actual_db_has_train_source": not cfg.EXCLUDE_TRAIN_FROM_DB}\n\n\ndef aggregate_query_reports(path, db, cfg):\n    db.execute("DROP TABLE IF EXISTS report_counts")\n    db.execute("CREATE TABLE report_counts(split TEXT,mode TEXT,ppm INT,n INT,lo REAL,hi REAL,u INT)")\n    for batch in pq.ParquetFile(path).iter_batches(cfg.BATCH_ROWS):\n        frame = batch.to_pandas().drop(columns="molecule_id")\n        db.executemany("INSERT INTO report_counts VALUES (?,?,?,?,?,?,?)", frame.itertuples(index=False, name=None))\n    db.execute("CREATE INDEX report_group ON report_counts(split,mode,ppm,n)")\n    result = []\n    groups = db.execute("SELECT DISTINCT split,mode,ppm FROM report_counts").fetchall()\n    for split, mode, ppm in groups:\n        args = (split, mode, ppm)\n        where = " WHERE split=? AND mode=? AND ppm=?"\n        n, mean, maximum, empty, lo, hi, unresolved = db.execute(\n            "SELECT COUNT(*),AVG(n),MAX(n),AVG(n=0),AVG(lo),AVG(hi),SUM(u) FROM report_counts" + where, args).fetchone()\n        row = dict(split=split, configuration=mode, ppm=ppm, queries=n, count_mean=mean, count_max=maximum, empty_fraction=empty)\n        for label, q in (("count_median", 0.5), ("count_p90", 0.9), ("count_p99", 0.99)):\n            pos = (n - 1) * q\n            i, j = math.floor(pos), math.ceil(pos)\n            values = db.execute("SELECT n FROM report_counts" + where + " ORDER BY n LIMIT ? OFFSET ?",\n                                args + (j - i + 1, i)).fetchall()\n            row[label] = values[0][0] + (values[-1][0] - values[0][0]) * (pos - i)\n        if split != "test":\n            row.update(oracle_mrr25_lower_bound=lo, oracle_mrr25_upper_bound=hi, unresolved_queries=unresolved)\n        result.append(row)\n    db.commit()\n    return result\n\n\ndef id_hash(db, split):\n    digest = hashlib.sha256()\n    for row in db.execute("SELECT id FROM molecules WHERE split3=? ORDER BY id", (split,)):\n        digest.update((row[0] + "\\n").encode())\n    return digest.hexdigest()\n\n\ndef create_synthetic_inputs(root):\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    smiles = ["CCO", "CC(=O)O", "c1ccccc1", "CCN", "CCC", "CCCO", "CCOC", "CCS"]\n    roles = ["train", "train", "train", "train", "val_model", "val_model", "val_rank", "val_rank"]\n    train, splits = [], []\n    for i, (smi, role) in enumerate(zip(smiles, roles)):\n        mol = Chem.MolFromSmiles(smi)\n        mass = float(rdMolDescriptors.CalcExactMolWt(mol))\n        ik14 = Chem.MolToInchiKey(mol)[:14]\n        for _ in range(60):\n            train.append(dict(molecule_id=f"smoke_{i}", smiles=smi, neutral_mass=mass,\n                              precursor_mz=mass+1.007276, adduct="[M+H]+", polarity="+"))\n        splits.append(dict(molecule_id=f"smoke_{i}", split3=role, ik14=ik14))\n    tests = [dict(molecule_id="test_a", neutral_mass=60.057514876),\n             dict(molecule_id="test_b", neutral_mass=46.041864812)]\n    for name, rows in (("train_clean", train), ("splits", splits), ("test_clean", tests)):\n        pq.write_table(pa.Table.from_pylist(rows), root/f"{name}.parquet")\n    print(f"Synthetic smoke inputs: {len(train)} train spectra; molecule counts={len(splits)}/2")\n    return root\n\n\ndef validate_artifacts(cfg):\n    state = context()\n    n = state["meta"]["rows"]\n    file = state["parquet"]\n    assert file.metadata.num_rows == n\n    assert set(DB_COLUMNS).issubset(file.schema_arrow.names)\n    assert state["fp"].dtype == np.uint8 and state["fp"].shape == (n, state["meta"]["fp_bytes"])\n    assert len(state["mass"]) == len(state["order"]) == len(state["formula_ids"]) == n\n    previous = -np.inf\n    known = 0\n    for i in range(0, n, cfg.BATCH_ROWS):\n        masses = state["mass"][i:i+cfg.BATCH_ROWS]\n        assert masses[0] >= previous and np.all(np.diff(masses) >= 0)\n        previous = masses[-1]\n        assert np.array_equal(state["order"][i:i+len(masses)], np.arange(i, i+len(masses)))\n        known += int(np.count_nonzero(state["valid"][i:i+len(masses)]))\n    assert state["offsets"][0] == 0 and state["offsets"][-1] == n\n    aliases = state["index"].execute("SELECT COUNT(*) FROM aliases WHERE row_id<0 OR row_id>=?", (n,)).fetchone()[0]\n    assert aliases == 0\n    if n:\n        sample = fetch_rows([0]).iloc[0]\n        assert 0 in set(query_by_mass(float(sample.mass), 0).row_id)\n        assert 0 in set(query_by_formula(sample.formula).row_id)\n        assert get_fingerprints([0]).shape == (1, state["meta"]["fp_bytes"])\n    for name in state["meta"]["required_files"]:\n        assert (state["root"]/name).is_file(), name\n    print(f"Sanity checks passed: {n:,} rows, {known:,} materialized fingerprints")\n    return {"rows": n, "fp_materialized": known, "sanity": "passed"}\n', encoding='utf-8')

In [ ]:
(WORK / 'fprank.py').write_text('"""Stage 06: bounded fingerprint scoring and validation-only rank tuning.\n\nCall candidates.configure(..., exclusion_scope=\'train\') before use. Arrays are\naligned positionally with queries_df (one row per molecule). No fitted global\nstate; all scoring parameters live in the JSON-serializable cfg dictionary.\n"""\nfrom functools import lru_cache\nimport re\nimport hashlib\n\nimport numpy as np\nimport pandas as pd\n\nBYTE_LUT = np.unpackbits(np.arange(256, dtype=np.uint8)[:, None], axis=1,\n                         bitorder=\'little\').astype(np.float32)\nELEMENTS = (\'C\', \'H\', \'N\', \'O\', \'S\', \'P\', \'F\', \'Cl\', \'Br\', \'I\')\nFEATURES = (\'nb_ll\', \'cos_sim\', \'tan_sim\', \'mass_score\', \'elem_score\', \'src_score\', \'maccs_cos\', \'maccs_tan\')\nDEFAULT_CONFIG = dict(ppm=15.0, temperature=1.0, sigma_ppm=5.0,\n                      fallback_ppm=50.0, fallback_da=0.05, min_candidates=25,\n                      batch_rows=2048, top_n=256, limit=100, eps=1e-6,\n                      max_tune_queries=5000,\n                      canonical_limit=25,\n                      weights=[1., 1., 1., 0.1, 0.1, 0.1, 0.5, 0.5],\n                      feature_scales=[4096., 1., 1., 1., 10., 1., 1., 1.])\nOUTPUT_DTYPES = dict(row_id=\'int64\', ik14=\'string\', smiles=\'string\', formula=\'string\',\n                     mass=\'float64\', source_mask=\'uint64\', fast_key=\'string\', identity_kind=\'string\',\n                     **{name: \'float32\' for name in FEATURES}, ppm_err=\'float64\', elem_dist=\'float32\',\n                     score=\'float64\', scoring_source_mask=\'uint64\', canon_ik14=\'string\',\n                     molecule_id=\'string\', rank=\'int64\')\n\n\ndef output_schema():\n    """Explicit schema even when a query contains only null canonical tail keys."""\n    import pyarrow as pa\n    return pa.schema([(name, pa.string() if dtype == \'string\' else pa.from_numpy_dtype(dtype))\n                      for name, dtype in OUTPUT_DTYPES.items()])\n\n\ndef normalize_output(frame):\n    return frame.loc[:, list(OUTPUT_DTYPES)].astype(OUTPUT_DTYPES)\n\n\ndef config(cfg):\n    result = dict(DEFAULT_CONFIG, **cfg)\n    for key in (\'ppm\', \'temperature\', \'sigma_ppm\', \'fallback_ppm\', \'fallback_da\'):\n        if not np.isfinite(result[key]) or result[key] <= 0:\n            raise ValueError(f\'{key} must be positive and finite\')\n    for key in (\'batch_rows\', \'top_n\', \'limit\', \'min_candidates\', \'max_tune_queries\', \'canonical_limit\'):\n        if not isinstance(result[key], int) or result[key] < 1:\n            raise ValueError(f\'{key} must be a positive integer\')\n    if not 0 < result[\'eps\'] < 0.5:\n        raise ValueError(\'eps must be in (0, 0.5)\')\n    if result[\'limit\'] > result[\'top_n\']:\n        raise ValueError(\'limit exceeds top_n\')\n    if result[\'canonical_limit\'] < 25:\n        raise ValueError(\'canonical_limit must cover all 25 submission guesses\')\n    for key in (\'weights\', \'feature_scales\'):\n        x = np.asarray(result[key], dtype=float)\n        if x.shape == (6,):\n            pad_len = len(FEATURES) - 6\n            if key == \'weights\':\n                x = np.pad(x, (0, pad_len), constant_values=0.5)\n            else:\n                x = np.pad(x, (0, pad_len), constant_values=1.0)\n            result[key] = x.tolist()\n        elif x.shape != (len(FEATURES),) or not np.isfinite(x).all():\n            raise ValueError(f\'Invalid {key}\')\n    if np.any(np.asarray(result[\'feature_scales\']) <= 0):\n        raise ValueError(\'feature_scales must be positive\')\n    return result\n\n\n@lru_cache(maxsize=32768)\ndef formula_counts(formula):\n    parts = re.findall(r\'([A-Z][a-z]?)([0-9]*)\', str(formula))\n    counts = np.zeros(10, dtype=np.float32)\n    for symbol, number in parts:\n        if symbol in ELEMENTS:\n            counts[ELEMENTS.index(symbol)] += int(number or 1)\n    return counts\n\n\ndef unpack_fingerprints(packed):\n    packed = np.asarray(packed)\n    if packed.dtype != np.uint8 or packed.ndim != 2 or packed.shape[1] != 533:\n        raise ValueError(\'Expected uint8 packed Morgan4096 + MACCS167, width 533\')\n    return (BYTE_LUT[packed[:, :512]].reshape(-1, 4096),\n            BYTE_LUT[packed[:, 512:]].reshape(-1, 168)[:, :167])\n\n\ndef score_features(frame, packed, probabilities, elements, mass, cfg, *,\n                   bits=None, maccs_probabilities=None, maccs_bits=None):\n    """Raw feature columns; composite NB is normalized by 4096 by default."""\n    if bits is None or (maccs_probabilities is not None and maccs_bits is None):\n        b, m = unpack_fingerprints(packed)\n        if bits is None:\n            bits = b\n        if maccs_bits is None:\n            maccs_bits = m\n    p = np.clip(np.asarray(probabilities, dtype=np.float64), cfg[\'eps\'], 1-cfg[\'eps\'])\n    logits = np.log(p) - np.log1p(-p)\n    slope = np.asarray(cfg.get(\'platt_slope\', 1.), dtype=float)\n    intercept = np.asarray(cfg.get(\'platt_intercept\', 0.), dtype=float)\n    for value in (slope, intercept):\n        if value.shape not in ((), (4096,)) or not np.isfinite(value).all():\n            raise ValueError(\'Platt parameters must be finite scalars or 4096-vectors\')\n    logits = (logits * slope + intercept) / cfg[\'temperature\']\n    log_p = -np.logaddexp(0., -logits)\n    log_not_p = -np.logaddexp(0., logits)\n    p = np.exp(log_p).astype(np.float32)\n    reliability = np.asarray(cfg.get(\'bit_weights\', np.ones(4096)), dtype=np.float64)\n    if reliability.shape != (4096,) or not np.isfinite(reliability).all() or (reliability < 0).any():\n        raise ValueError(\'bit_weights must be 4096 finite nonnegative values\')\n    dot = bits @ p\n    count = bits.sum(axis=1)\n    nb = bits @ (reliability * logits).astype(np.float32) + np.sum(reliability * log_not_p)\n    ppm = np.abs(frame.mass.to_numpy(dtype=float) - mass) / mass * 1e6\n    elem = np.abs(np.stack([formula_counts(f) for f in frame.formula]) - elements).sum(axis=1)\n    # Curated-source prior excludes TRAIN_BIT and uses the filtered source mask.\n    src = np.array([(int(m) & ~1).bit_count() for m in frame.source_mask], dtype=float)\n\n    if maccs_probabilities is not None:\n        p_mc = np.clip(np.asarray(maccs_probabilities, dtype=np.float64), cfg[\'eps\'], 1 - cfg[\'eps\'])\n        mc_slope = np.asarray(cfg.get(\'maccs_platt_slope\', 1.), dtype=float)\n        mc_intercept = np.asarray(cfg.get(\'maccs_platt_intercept\', 0.), dtype=float)\n        mc_logits = (np.log(p_mc) - np.log1p(-p_mc)) * mc_slope + mc_intercept\n        mc_logits = mc_logits / cfg[\'temperature\']\n        mc_log_p = -np.logaddexp(0., -mc_logits)\n        p_mc = np.exp(mc_log_p).astype(np.float32)\n        dot_mc = maccs_bits @ p_mc\n        count_mc = maccs_bits.sum(axis=1)\n        norm_mc = np.linalg.norm(p_mc)\n        maccs_cos = (dot_mc / np.maximum(np.sqrt(count_mc) * norm_mc, 1e-12)).astype(np.float32)\n        maccs_tan = (dot_mc / np.maximum(count_mc + p_mc.sum() - dot_mc, 1e-12)).astype(np.float32)\n    else:\n        maccs_cos = np.zeros(len(frame), dtype=np.float32)\n        maccs_tan = np.zeros(len(frame), dtype=np.float32)\n\n    return np.column_stack((nb, dot / np.maximum(np.sqrt(count)*np.linalg.norm(p), 1e-12),\n                            dot / np.maximum(count+p.sum()-dot, 1e-12),\n                            -0.5*(ppm/cfg[\'sigma_ppm\'])**2, -elem, src,\n                            maccs_cos, maccs_tan)).astype(np.float32)\n\n\ndef validate_inputs(queries, fp, elem, maccs=None):\n    if not {\'molecule_id\', \'neutral_mass\'} <= set(queries):\n        raise ValueError(\'queries require molecule_id and observed neutral_mass\')\n    if queries.molecule_id.isna().any() or queries.molecule_id.astype(str).duplicated().any():\n        raise ValueError(\'Expected unique, non-null molecule IDs\')\n    if fp.shape != (len(queries), 4096) or elem.shape != (len(queries), 10):\n        raise ValueError(\'Prediction shape/alignment mismatch\')\n    if maccs is not None and maccs.shape != (len(queries), 167):\n        raise ValueError(f\'MACCS prediction shape mismatch, expected ({len(queries)}, 167), got {maccs.shape}\')\n    masses = queries.neutral_mass.to_numpy(dtype=float)\n    if not np.isfinite(masses).all() or (masses <= 0).any():\n        raise ValueError(\'Invalid observed neutral mass\')\n    # Chunk validation avoids copying an entire prediction mmap.\n    for start in range(0, len(queries), 1024):\n        p, e = fp[start:start+1024], elem[start:start+1024]\n        if not np.isfinite(p).all() or not np.isfinite(e).all() or (p < 0).any() or (p > 1).any():\n            raise ValueError(\'Nonfinite predictions or probabilities outside [0,1]\')\n        if maccs is not None:\n            mc = maccs[start:start+1024]\n            if not np.isfinite(mc).all() or (mc < 0).any() or (mc > 1).any():\n                raise ValueError(\'Nonfinite MACCS predictions or probabilities outside [0,1]\')\n\n\ndef exclusion_ids(queries, cfg):\n    import candidates\n    if candidates.context()[\'scope\'] != \'train\':\n        raise ValueError("Configure candidates with exclusion_scope=\'train\'")\n    is_validation = (\'split3\' in queries and queries.split3.isin([\'val_model\', \'val_rank\']).any())\n    if is_validation and \'exclude_keys\' not in cfg:\n        raise ValueError(\'Validation requires validation_exclusions(db) in cfg[exclude_keys]\')\n    return candidates.excluded_rows(cfg.get(\'exclude_keys\', ()))\n\n\ndef candidate_batches(mass, ppm, blocked, cfg, *, outside_ppm=None):\n    import candidates\n    lo, hi = candidates.mass_bounds(mass, ppm)\n    state = candidates.context()\n    size = min(cfg[\'batch_rows\'], state[\'meta\'][\'max_materialized_rows\'])\n    spans = [(lo, hi)]\n    if outside_ppm is not None:\n        inner_lo, inner_hi = candidates.mass_bounds(mass, outside_ppm)\n        spans = [(lo, min(hi, inner_lo)), (max(lo, inner_hi), hi)]\n    for first, last in spans:\n        for start in range(first, last, size):\n            ids = state[\'order\'][start:min(last, start+size)]\n            # Resolve exclude_keys once per call, not once per query or chunk.\n            frame = candidates.filter_sources(candidates.fetch_rows(ids), excluded_ids=blocked)\n            if not frame.empty:\n                yield frame\n\n\ndef query_features(mass, p, e, blocked, cfg, *, maccs=None):\n    import candidates\n    count = 0\n    for frame in candidate_batches(mass, cfg[\'ppm\'], blocked, cfg):\n        count += len(frame)\n        yield frame, score_features(frame, candidates.get_fingerprints(frame.row_id, smiles=frame.smiles.to_numpy()),\n                                     p, e, mass, cfg, maccs_probabilities=maccs)\n    if count < cfg[\'min_candidates\']:\n        wide = max(cfg[\'ppm\'], cfg[\'fallback_ppm\'], cfg[\'fallback_da\']/mass*1e6)\n        for frame in candidate_batches(mass, wide, blocked, cfg, outside_ppm=cfg[\'ppm\']):\n            count += len(frame)\n            yield frame, score_features(frame, candidates.get_fingerprints(frame.row_id, smiles=frame.smiles.to_numpy()),\n                                         p, e, mass, cfg, maccs_probabilities=maccs)\n    if count == 0:\n        for frame in nearest_candidate_batches(mass, blocked, cfg):\n            yield frame, score_features(frame, candidates.get_fingerprints(frame.row_id, smiles=frame.smiles.to_numpy()),\n                                         p, e, mass, cfg, maccs_probabilities=maccs)\n\n\ndef nearest_candidate_batches(mass, blocked, cfg, limit=50):\n    """Scan out from searchsorted until the nearest 50 eligible rows are found.\n\n    Source filtering is identical to window retrieval. At least one eligible DB\n    row is necessary for coverage; an empty/fully excluded database fails clearly.\n    """\n    import candidates\n    state = candidates.context()\n    masses, order = state[\'mass\'], state[\'order\']\n    right = int(np.searchsorted(masses, mass))\n    left, found = right-1, 0\n    size = min(cfg[\'batch_rows\'], state[\'meta\'][\'max_materialized_rows\'], limit)\n    while found < limit and (left >= 0 or right < len(masses)):\n        ids = []\n        for _ in range(min(size, limit-found)):\n            if left < 0 and right >= len(masses):\n                break\n            if right >= len(masses) or (left >= 0 and mass-masses[left] <= masses[right]-mass):\n                ids.append(int(order[left]))\n                left -= 1\n            else:\n                ids.append(int(order[right]))\n                right += 1\n        frame = candidates.filter_sources(candidates.fetch_rows(ids), excluded_ids=blocked)\n        if not frame.empty:\n            found += len(frame)\n            yield frame\n    if found == 0:\n        raise ValueError(\'Candidate database has no eligible rows, including nearest-mass fallback\')\n\n\ndef composite(features, cfg):\n    w = np.asarray(cfg[\'weights\'], dtype=float)\n    s = np.asarray(cfg[\'feature_scales\'], dtype=float)\n    k = features.shape[1]\n    return features @ (w[:k] / s[:k])\n\n\ndef top_indices(scores, row_ids, n):\n    # Deterministic tie-breaking also at the shortlist boundary.\n    return np.lexsort((row_ids, -scores))[:n]\n\n\ndef iter_rank_candidates(queries_df, fp_probs, elem_preds, cfg, maccs_preds=None):\n    """Stream rows and flush the lazy chemistry transaction at each call boundary."""\n    import candidates\n    try:\n        yield from _iter_rank_candidates(queries_df, fp_probs, elem_preds, cfg, maccs_preds=maccs_preds)\n    finally:\n        candidates.flush_cache()\n\n\ndef _iter_rank_candidates(queries_df, fp_probs, elem_preds, cfg, maccs_preds=None):\n    """Yield one top-100 frame per query in ascending observed-mass order."""\n    import candidates\n    cfg = config(cfg)\n    validate_inputs(queries_df, fp_probs, elem_preds, maccs=maccs_preds)\n    blocked = exclusion_ids(queries_df, cfg)\n    for i in np.argsort(queries_df.neutral_mass.to_numpy(), kind=\'stable\'):\n        query = queries_df.iloc[i]\n        shortlist = None\n        mc_p = maccs_preds[i] if maccs_preds is not None else None\n        for frame, features in query_features(float(query.neutral_mass), fp_probs[i], elem_preds[i],\n                                              blocked, cfg, maccs=mc_p):\n            frame = frame.copy()\n            for j, name in enumerate(FEATURES):\n                frame[name] = features[:, j]\n            frame[\'ppm_err\'] = np.abs(frame.mass-float(query.neutral_mass))/float(query.neutral_mass)*1e6\n            frame[\'elem_dist\'] = -frame.elem_score\n            frame[\'score\'] = composite(features, cfg)\n            frame[\'scoring_source_mask\'] = frame.source_mask\n            merged = pd.concat([shortlist, frame], ignore_index=True) if shortlist is not None else frame\n            shortlist = merged.iloc[top_indices(merged.score.to_numpy(), merged.row_id.to_numpy(), cfg[\'top_n\'])]\n        if shortlist is None:\n            raise RuntimeError(f\'No eligible candidates for {query.molecule_id}\')\n        final = candidates.finalize_candidates(shortlist, top_n=cfg[\'top_n\'], limit=cfg[\'limit\'],\n                                                canonical_limit=cfg[\'canonical_limit\'])\n        final = final.drop(columns=[\'member_row_ids\'], errors=\'ignore\')\n        final[\'source_mask\'] = final.source_mask.astype(np.uint64)\n        final[\'scoring_source_mask\'] = final.scoring_source_mask.astype(np.uint64)\n        final[\'molecule_id\'] = str(query.molecule_id)\n        final[\'rank\'] = np.arange(1, len(final)+1)\n        yield normalize_output(final)\n\n\ndef rank_candidates(queries_df, fp_probs, elem_preds, cfg, maccs_preds=None):\n    """Stateless Stage 10 API. Use iter_rank_candidates for bounded output RAM."""\n    frames = list(iter_rank_candidates(queries_df, fp_probs, elem_preds, cfg, maccs_preds=maccs_preds))\n    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()\n\n\ndef tuning_pool(mass, blocked, cfg):\n    """One 20-ppm retrieval; cache wider fallback only if a grid window needs it."""\n    max_ppm = 20.0\n    count, narrow_count = 0, 0\n    for frame in candidate_batches(mass, max_ppm, blocked, cfg):\n        count += len(frame)\n        narrow_count += int((np.abs(frame.mass.to_numpy()-mass)/mass*1e6 <= 10.).sum())\n        yield frame\n    if narrow_count < cfg[\'min_candidates\']:\n        wide = max(max_ppm, cfg[\'fallback_ppm\'], cfg[\'fallback_da\']/mass*1e6)\n        for frame in candidate_batches(mass, wide, blocked, cfg, outside_ppm=max_ppm):\n            count += len(frame)\n            yield frame\n    if count == 0:\n        yield from nearest_candidate_batches(mass, blocked, cfg)\n\n\ndef build_tuning_cache(queries, fp, elem, truth_keys, cfg, indices, maccs=None):\n    """In-memory small feature arrays, no packed/unpacked fingerprints retained.\n\n    Candidate metadata/fingerprints are read once per sampled query. Hit labels\n    use existing fast/canonical keys (a proxy, without tautomer canonicalization).\n    An optional query truth_fast_key supports truths with a different tautomer key.\n    """\n    import candidates\n    blocked = exclusion_ids(queries, cfg)\n    chunks = []\n    for q, i in enumerate(indices):\n        mass = float(queries.iloc[i].neutral_mass)\n        truth = truth_keys[i]\n        fast_truth = queries.iloc[i].get(\'truth_fast_key\', truth)\n        mc_i = maccs[i] if maccs is not None else None\n        for frame in tuning_pool(mass, blocked, cfg):\n            packed = candidates.get_fingerprints(frame.row_id, smiles=frame.smiles.to_numpy())\n            bits, cand_maccs = unpack_fingerprints(packed)\n            features = np.stack([score_features(frame, packed, fp[i], elem[i], mass,\n                                                dict(cfg, temperature=t), bits=bits,\n                                                maccs_probabilities=mc_i, maccs_bits=cand_maccs)\n                                 for t in (0.7, 1., 1.3)], axis=1)\n            hit = (frame.fast_key.eq(truth) | frame.fast_key.eq(fast_truth) | frame.ik14.eq(truth))\n            chunks.append(dict(x=features, row_ids=frame.row_id.to_numpy(dtype=np.int64),\n                               is_hit=hit.to_numpy(dtype=bool),\n                               ppm_err=np.abs(frame.mass.to_numpy(dtype=float)-mass)/mass*1e6,\n                               query=np.full(len(frame), q, dtype=np.int32)))\n        if (q+1) % 100 == 0:\n            print(f\'Tuning features: {q+1}/{len(indices)} queries\', flush=True)\n    cache = {key: np.concatenate([chunk[key] for chunk in chunks]) for key in chunks[0]}\n    cache[\'n_queries\'] = len(indices)\n    return cache\n\n\ndef tuning_window(cache, ppm, temperature_index, cfg):\n    """Filter each query\'s cached pool; sparse primary windows retain fallback rows."""\n    within = cache[\'ppm_err\'] <= ppm\n    counts = np.bincount(cache[\'query\'][within], minlength=cache[\'n_queries\'])\n    keep = within | (counts[cache[\'query\']] < cfg[\'min_candidates\'])\n    return dict(x=cache[\'x\'][keep, temperature_index, :],\n                row_ids=cache[\'row_ids\'][keep], is_hit=cache[\'is_hit\'][keep],\n                query=cache[\'query\'][keep], n_queries=cache[\'n_queries\'])\n\n\ndef cache_reciprocal_ranks(cache, cfg):\n    """Vectorized raw-key MRR@25 proxy, including misses; no I/O or full sorting.\n\n    Count candidates ahead of the highest-scoring hit with row_id tie-breaking.\n    Official canonical deduplicated MRR is measured on full outputs after freeze.\n    """\n    scores = composite(cache[\'x\'], cfg)\n    q, hit, ids, n = cache[\'query\'], cache[\'is_hit\'], cache[\'row_ids\'], cache[\'n_queries\']\n    best = np.full(n, -np.inf)\n    np.maximum.at(best, q[hit], scores[hit])\n    best_ids = np.full(n, np.iinfo(np.int64).max, dtype=np.int64)\n    top_hits = hit & (scores == best[q])\n    np.minimum.at(best_ids, q[top_hits], ids[top_hits])\n    ahead = (scores > best[q]) | ((scores == best[q]) & (ids < best_ids[q]))\n    ranks = 1 + np.bincount(q[ahead], minlength=n)\n    valid = np.isfinite(best) & (ranks <= 25)\n    rr = np.zeros(n, dtype=float)\n    rr[valid] = 1. / ranks[valid]\n    return rr\n\n\ndef cache_mrr(cache, cfg):\n    rr = cache_reciprocal_ranks(cache, cfg)\n    return float(rr.mean()) if len(rr) else 0.0\n\n\ndef tuning_diagnostics(window, cfg, hit_in_20ppm):\n    """Conditional numerator and denominator use the same fixed 20-ppm hit cohort."""\n    rr = cache_reciprocal_ranks(window, cfg)\n    covered = int(hit_in_20ppm.sum())\n    return dict(overall_proxy_mrr25=float(rr.mean()) if len(rr) else 0.0,\n                covered_hit_mrr25=float(rr[hit_in_20ppm].mean()) if covered else 0.0,\n                hit_queries_20ppm=covered,\n                oracle_20ppm_recall=covered / max(len(rr), 1))\n\n\ndef tune(queries, fp, elem, truth_keys, cfg, cache_dir=None, sweeps=3,\n         max_tune_queries: int = 5000, maccs=None):\n    """Deterministic sample, one retrieval pass, then entirely in-memory search.\n\n    cache_dir is retained for call compatibility and is unused. cfg may lower the\n    sample cap further. No canonicalization or candidate I/O occurs during search.\n    """\n    cfg = config(cfg)\n    validate_inputs(queries, fp, elem, maccs=maccs)\n    if \'split3\' not in queries or not queries.split3.eq(\'val_model\').all():\n        raise ValueError(\'Tuning may consume val_model only\')\n    if len(queries) == 0 or len(truth_keys) != len(queries):\n        raise ValueError(\'Nonempty, aligned validation truths required\')\n    if not isinstance(max_tune_queries, int) or max_tune_queries < 1:\n        raise ValueError(\'max_tune_queries must be a positive integer\')\n    cap = min(max_tune_queries, cfg[\'max_tune_queries\'])\n    indices = np.random.default_rng(42).choice(len(queries), min(cap, len(queries)), replace=False)\n    indices = indices[np.argsort(queries.neutral_mass.to_numpy()[indices], kind=\'stable\')]\n    import candidates\n    try:\n        if maccs is not None:\n            cache = build_tuning_cache(queries, fp, elem, truth_keys, cfg, indices, maccs=maccs)\n        else:\n            cache = build_tuning_cache(queries, fp, elem, truth_keys, cfg, indices)\n    finally:\n        candidates.flush_cache()\n    hit_in_20ppm = np.zeros(cache[\'n_queries\'], dtype=bool)\n    hit_in_20ppm[cache[\'query\'][cache[\'is_hit\'] & (cache[\'ppm_err\'] <= 20.)]] = True\n    best, best_score, history = None, -1., []\n    for ppm in (10., 15., 20.):\n        for t, temperature in enumerate((0.7, 1., 1.3)):\n            trial = dict(cfg, ppm=ppm, temperature=temperature)\n            window = tuning_window(cache, ppm, t, cfg)\n            current = cache_mrr(window, trial)\n            for _ in range(sweeps):\n                improved = False\n                for j in range(len(trial[\'weights\'])):\n                    origin = trial[\'weights\'][j]\n                    values = (0.0, origin*0.5, origin*0.75, origin*1.25, max(origin*2.0, 0.1))\n                    # Prefer the smaller step if several proposals improve MRR equally.\n                    for value in sorted(set(values), key=lambda v: (abs(v-origin), v)):\n                        weights = list(trial[\'weights\'])\n                        weights[j] = value\n                        proposed = dict(trial, weights=weights)\n                        score = cache_mrr(window, proposed)\n                        if score > current + 1e-12:\n                            trial, current, improved = proposed, score, True\n                if not improved:\n                    break\n            diagnostics = tuning_diagnostics(window, trial, hit_in_20ppm)\n            history.append(dict(ppm=ppm, temperature=temperature, mrr25=current,\n                                weights=trial[\'weights\'], **diagnostics))\n            print(\'Tuning:\', history[-1], flush=True)\n            if current > best_score:\n                best, best_score = dict(trial), current\n                best_diagnostics = diagnostics\n    best.pop(\'exclude_keys\', None)\n    ids = queries.iloc[indices].molecule_id.astype(str).tolist()\n    best.update(tuned_on=\'val_model\', tuning_proxy_mrr25=best_score, search_history=history,\n                **best_diagnostics,\n                tuning_sweeps=sweeps, covered_hit_scope=\'truth present in strict 20-ppm pool\',\n                tuning_metric=\'raw fast_key/ik14 MRR@25 proxy (no tautomer deduplication)\',\n                max_tune_queries=cap, tuning_queries=len(indices), tuning_seed=42,\n                tuning_ids_hash=hashlib.sha256(\'\'.join(sorted(ids)).encode()).hexdigest(),\n                calibration=\'Stage05 probabilities; temperature selected on val_model\')\n    return best\n', encoding='utf-8')

In [ ]:
(WORK / 'spec2fp.py').write_text('"""spec2fp.py - Spectrum-to-Fingerprint model, tensorization, and inference.\n\nDefines the Spec2FP Transformer, peak tensorization, and predict() API.\nImport ONLY from /kaggle/working/.\n"""\nimport json, math, sys, warnings, gc\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple, Union, Any\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nADDUCT_VOCAB = [\n    "<pad>", "[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+",\n    "[M+H-H2O]+", "[M+FA-H]-", "[M+Cl]-", "[M]+", "[M-H2O+H]+",\n    "[M-H2O-H]-", "[M+2H]2+", "[M-2H]2-", "[M+HAc-H]-",\n    "[M+2Na-H]+", "[M+CH2O2-H]-", "<unk>",\n]\nADDUCT2IDX = {a: i for i, a in enumerate(ADDUCT_VOCAB)}\nINST_VOCAB = [\n    "<pad>", "timsTOF", "qtof", "orbitrap", "FTICR", "QqQ",\n    "ion_trap", "None/Unknown", "<unk>",\n]\nINST2IDX = {a: i for i, a in enumerate(INST_VOCAB)}\nPOLARITY_VOCAB = ["<pad>", "positive", "negative", "<unk>"]\nPOLARITY2IDX = {a: i for i, a in enumerate(POLARITY_VOCAB)}\nELEMENTS_ORDER = ["C", "H", "N", "O", "S", "P", "F", "Cl", "Br", "I"]\nN_ELEMENTS = len(ELEMENTS_ORDER)\n\n\ndef parse_peak_array(val: Any) -> np.ndarray:\n    """Robustly parse peak m/z or intensity into a float64 numpy array."""\n    if isinstance(val, np.ndarray):\n        return val.astype(np.float64)\n    if isinstance(val, (list, tuple)):\n        return np.asarray(val, dtype=np.float64)\n    if isinstance(val, str):\n        try:\n            return np.asarray(json.loads(val), dtype=np.float64)\n        except Exception:\n            pass\n    try:\n        return np.asarray(val, dtype=np.float64)\n    except Exception:\n        return np.zeros(0, dtype=np.float64)\n\n\ndef sinusoidal_embed(values: np.ndarray, dim: int = 64) -> np.ndarray:\n    """Multi-scale sinusoidal encoding resolving 0.005 Da mass defects up to 2000 Da."""\n    freqs = np.logspace(\n        math.log10(2 * math.pi / 2000.0),\n        math.log10(2 * math.pi / 0.005),\n        dim // 2, dtype=np.float32)\n    angles = np.outer(values.astype(np.float32), freqs)\n    return np.concatenate([np.sin(angles), np.cos(angles)], axis=-1)\n\n\ndef tensorize_spectrum(\n    mz: Any, inten: Any, precursor_mz: float,\n    adduct: Any, ce: float, instrument: Any, polarity: Any,\n    top_k: int = 192, mz_dim: int = 64,\n) -> Dict[str, np.ndarray]:\n    """Convert a single spectrum into model-ready numpy arrays."""\n    mz = parse_peak_array(mz)\n    inten = parse_peak_array(inten)\n    mask_valid = mz <= precursor_mz + 5.0\n    mz, inten = mz[mask_valid], inten[mask_valid]\n    if len(mz) == 0:\n        mz = np.array([precursor_mz], dtype=np.float64)\n        inten = np.array([1.0], dtype=np.float64)\n    max_int = max(inten.max(), 1e-12)\n    inten = np.sqrt(inten / max_int)\n    if len(mz) > top_k:\n        idx = np.argsort(inten)[-top_k:]\n        idx = idx[np.argsort(mz[idx])]\n        mz, inten = mz[idx], inten[idx]\n    n = len(mz)\n    mz_emb = sinusoidal_embed(mz, mz_dim)\n    nl = precursor_mz - mz\n    nl_emb = sinusoidal_embed(nl, mz_dim)\n    log_int = np.log1p(inten).astype(np.float32).reshape(-1, 1)\n    peak_feat = np.concatenate([mz_emb, log_int, nl_emb], axis=1).astype(np.float32)\n    adduct_idx = int(adduct) if isinstance(adduct, (int, np.integer)) else ADDUCT2IDX.get(str(adduct).strip(), ADDUCT2IDX["<unk>"])\n    inst_idx = int(instrument) if isinstance(instrument, (int, np.integer)) else INST2IDX.get(str(instrument).strip(), INST2IDX["<unk>"])\n    pol_idx = int(polarity) if isinstance(polarity, (int, np.integer)) else POLARITY2IDX.get(str(polarity).strip().lower(), POLARITY2IDX["<unk>"])\n    ce_val = float(ce) if not (ce is None or (isinstance(ce, float) and math.isnan(ce))) else 0.0\n    return {\n        "peak_feat": peak_feat, "n_peaks": n,\n        "adduct_idx": adduct_idx, "inst_idx": inst_idx, "pol_idx": pol_idx,\n        "ce_val": np.float32(ce_val), "prec_mz": np.float32(precursor_mz),\n    }\n\n\ndef collate_spectra(batch: list, top_k: int = 192, feat_dim: int = 129):\n    """Collate list of tensorize_spectrum outputs into batched tensors."""\n    B = len(batch)\n    max_len = top_k + 3\n    peaks = torch.zeros(B, max_len, feat_dim, dtype=torch.float32)\n    mask = torch.ones(B, max_len, dtype=torch.bool)\n    adducts = torch.zeros(B, dtype=torch.long)\n    instruments = torch.zeros(B, dtype=torch.long)\n    polarities = torch.zeros(B, dtype=torch.long)\n    ces = torch.zeros(B, dtype=torch.float32)\n    prec_mzs = torch.zeros(B, dtype=torch.float32)\n    for i, item in enumerate(batch):\n        n = item["n_peaks"]\n        peaks[i, :n] = torch.from_numpy(item["peak_feat"][:n])\n        mask[i, :n] = False\n        adducts[i] = item["adduct_idx"]\n        instruments[i] = item["inst_idx"]\n        polarities[i] = item["pol_idx"]\n        ces[i] = float(item["ce_val"])\n        prec_mzs[i] = float(item["prec_mz"])\n    return {\n        "peaks": peaks, "pad_mask": mask,\n        "adduct": adducts, "instrument": instruments, "polarity": polarities,\n        "ce": ces, "prec_mz": prec_mzs,\n    }\n\n\nclass Spec2FPModel(nn.Module):\n    """Transformer encoder mapping MS/MS spectrum to molecular fingerprint."""\n\n    def __init__(self, cfg: dict):\n        super().__init__()\n        self.cfg = cfg\n        d, n_layers, n_heads = cfg.get("d_model", 320), cfg.get("n_layers", 5), cfg.get("n_heads", 8)\n        dropout, feat_dim = cfg.get("dropout", 0.1), cfg.get("feat_dim", 129)\n        fp_bits, maccs_bits = cfg.get("fp_bits", 4096), cfg.get("maccs_bits", 167)\n        n_elements = cfg.get("n_elements", N_ELEMENTS)\n        self.peak_proj = nn.Sequential(nn.Linear(feat_dim, d), nn.GELU(), nn.LayerNorm(d), nn.Dropout(dropout))\n        self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)\n        self.adduct_emb = nn.Embedding(len(ADDUCT_VOCAB), d, padding_idx=0)\n        self.inst_emb = nn.Embedding(len(INST_VOCAB), d, padding_idx=0)\n        self.pol_emb = nn.Embedding(len(POLARITY_VOCAB), d, padding_idx=0)\n        self.ce_proj = nn.Sequential(nn.Linear(1, d), nn.GELU())\n        self.prec_proj = nn.Sequential(nn.Linear(1, d), nn.GELU())\n        enc_layer = nn.TransformerEncoderLayer(\n            d_model=d, nhead=n_heads, dim_feedforward=d * 4,\n            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)\n        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)\n        self.norm = nn.LayerNorm(d)\n        self.fp_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, fp_bits))\n        self.maccs_head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Linear(d // 2, maccs_bits))\n        self.heavy_atom_head = nn.Linear(d, 1)\n        self.ring_count_head = nn.Linear(d, 1)\n        self.elem_presence_head = nn.Linear(d, n_elements)\n        self.elem_count_head = nn.Linear(d, n_elements)\n\n    def forward(self, peaks, pad_mask, adduct, instrument, polarity, ce, prec_mz):\n        B = peaks.size(0)\n        x = self.peak_proj(peaks)\n        cls = self.cls_token.expand(B, -1, -1)\n        meta1 = (\n            self.adduct_emb(adduct)\n            + self.ce_proj(ce.unsqueeze(-1) / 100.0)\n            + self.prec_proj(prec_mz.unsqueeze(-1) / 1000.0)\n        )\n        meta2 = self.inst_emb(instrument) + self.pol_emb(polarity)\n        meta = torch.stack([meta1, meta2], dim=1)\n        x = torch.cat([cls, meta, x], dim=1)\n        cls_meta_mask = torch.zeros(B, 3, dtype=torch.bool, device=pad_mask.device)\n        full_mask = torch.cat([cls_meta_mask, pad_mask], dim=1)\n        x = self.encoder(x, src_key_padding_mask=full_mask)\n        cls_out = x[:, 0]\n        valid_tok = (~full_mask).float().unsqueeze(-1)\n        mean_out = (x * valid_tok).sum(dim=1) / valid_tok.sum(dim=1).clamp(min=1.0)\n        x = self.norm(cls_out + mean_out)\n        return {\n            "fp_logits": self.fp_head(x), "maccs_logits": self.maccs_head(x),\n            "heavy_atoms": self.heavy_atom_head(x).squeeze(-1),\n            "ring_count": self.ring_count_head(x).squeeze(-1),\n            "elem_presence": self.elem_presence_head(x),\n            "elem_count": self.elem_count_head(x),\n        }\n\n\ndef load_model(config_path: str, weights_path: str, device: str = "cpu"):\n    """Load Spec2FP model from config + weights."""\n    with open(config_path) as f:\n        cfg = json.load(f)\n    model = Spec2FPModel(cfg["model"])\n    sd = torch.load(weights_path, map_location=device, weights_only=False)\n    if "ema_state_dict" in sd:\n        model.load_state_dict({k.replace("module.", ""): v for k, v in sd["ema_state_dict"].items()})\n    elif "model_state_dict" in sd:\n        model.load_state_dict({k.replace("module.", ""): v for k, v in sd["model_state_dict"].items()})\n    else:\n        model.load_state_dict({k.replace("module.", ""): v for k, v in sd.items()})\n    pw_path = Path(weights_path).parent / "spec2fp_pos_weights.npy"\n    if pw_path.is_file():\n        try:\n            model.pos_weights = np.load(pw_path)\n        except Exception:\n            pass\n    model.eval()\n    return model.to(device), cfg\n\n\n@torch.no_grad()\ndef predict_full(model, clean_df, device="cuda", batch_size=128, top_k=192, mz_dim=64, pos_weights=None):\n    """Predict Morgan-4096, MACCS-167, and element counts with streaming molecule-level pooling."""\n    raw_model = model.module if hasattr(model, "module") else model\n    raw_model.eval()\n    pw = pos_weights if pos_weights is not None else getattr(raw_model, "pos_weights", None)\n    pw_shift = 0.5 * torch.log(torch.as_tensor(pw, device=device, dtype=torch.float32).clamp(min=1.0)) if pw is not None else None\n    feat_dim, n_rows = 2 * mz_dim + 1, len(clean_df)\n    if n_rows == 0:\n        return (np.zeros((0, 4096), dtype=np.float16), np.zeros((0, 167), dtype=np.float16),\n                np.zeros((0, N_ELEMENTS), dtype=np.float32), [])\n\n    mol_ids_arr = clean_df["molecule_id"].to_numpy().astype(str)\n    unique_mols, inv_indices = np.unique(mol_ids_arr, return_inverse=True)\n    n_unique = len(unique_mols)\n    fp_sum = np.zeros((n_unique, 4096), dtype=np.float32)\n    maccs_sum = np.zeros((n_unique, 167), dtype=np.float32)\n    elem_sum = np.zeros((n_unique, N_ELEMENTS), dtype=np.float32)\n    counts = np.zeros(n_unique, dtype=np.float32)\n\n    mz_col = clean_df["mz"].to_numpy()\n    int_col = clean_df["inten"].to_numpy()\n    pmz_col = clean_df["precursor_mz"].to_numpy(dtype=np.float32)\n    add_col = clean_df["adduct"].to_numpy() if "adduct" in clean_df.columns else np.array(["[M+H]+"] * n_rows)\n    ce_col = pd.to_numeric(clean_df["ce"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32) if "ce" in clean_df.columns else np.zeros(n_rows, dtype=np.float32)\n    inst_col = clean_df["instrument"].to_numpy() if "instrument" in clean_df.columns else np.array(["unknown"] * n_rows)\n    pol_col = clean_df["polarity"].to_numpy() if "polarity" in clean_df.columns else np.array(["positive"] * n_rows)\n    dev_type, is_cuda = ("cuda" if "cuda" in str(device) else "cpu"), ("cuda" in str(device))\n\n    for s_idx in range(0, n_rows, batch_size):\n        e_idx = min(s_idx + batch_size, n_rows)\n        items = [\n            tensorize_spectrum(\n                mz_col[r], int_col[r], float(pmz_col[r]),\n                str(add_col[r]), float(ce_col[r]),\n                str(inst_col[r]), str(pol_col[r]), top_k=top_k, mz_dim=mz_dim)\n            for r in range(s_idx, e_idx)\n        ]\n        bd = collate_spectra(items, top_k=top_k, feat_dim=feat_dim)\n        bd = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in bd.items()}\n        with torch.amp.autocast(device_type=dev_type, dtype=torch.float16, enabled=is_cuda):\n            out = raw_model(**bd)\n        fp_logits = out["fp_logits"] - pw_shift if pw_shift is not None else out["fp_logits"]\n        batch_fp = torch.sigmoid(fp_logits).float().cpu().numpy()\n        batch_maccs = torch.sigmoid(out["maccs_logits"]).float().cpu().numpy()\n        batch_elem = out["elem_count"].float().cpu().numpy()\n        batch_inv = inv_indices[s_idx:e_idx]\n        np.add.at(fp_sum, batch_inv, batch_fp)\n        np.add.at(maccs_sum, batch_inv, batch_maccs)\n        np.add.at(elem_sum, batch_inv, batch_elem)\n        np.add.at(counts, batch_inv, 1.0)\n\n    counts_col = np.maximum(counts, 1.0)[:, None]\n    return ((fp_sum / counts_col).astype(np.float16),\n            (maccs_sum / counts_col).astype(np.float16),\n            (elem_sum / counts_col).astype(np.float32), unique_mols.tolist())\n\n\n@torch.no_grad()\ndef predict(model, clean_df, device="cuda", batch_size=128, top_k=192, mz_dim=64, pos_weights=None, return_maccs=False):\n    """Predict fingerprints with optional MACCS return for backward-compatibility."""\n    fp_pred, maccs_pred, elem_pred, mol_ids = predict_full(\n        model, clean_df, device=device, batch_size=batch_size, top_k=top_k, mz_dim=mz_dim, pos_weights=pos_weights\n    )\n    if return_maccs:\n        return fp_pred, maccs_pred, elem_pred, mol_ids\n    return fp_pred, elem_pred, mol_ids\n', encoding='utf-8')

In [ ]:
def import_work(name):
    path = WORK / (name + '.py')
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    assert Path(module.__file__).resolve() == path.resolve()
    return module
evalkit = import_work('evalkit')
candidates = import_work('candidates')
spec2fp = import_work('spec2fp')
fprank = import_work('fprank')

def locate(filename, directory=None):
    if directory is not None:
        path = Path(directory) / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    paths = sorted({p.resolve() for root in (WORK, Path('/kaggle/input'))
                    for p in root.rglob(filename) if p.is_file()},
                   key=lambda p: (p.is_relative_to(WORK.resolve()), p.stat().st_mtime, str(p)),
                   reverse=True)
    if not paths:
        raise FileNotFoundError(f'Missing {filename} in /kaggle/working or /kaggle/input')
    return paths[0]

def heldout_exclusions(split_path, train_path):
    def read_metadata(path, wanted):
        available = pq.ParquetFile(path).schema_arrow.names
        return pd.read_parquet(path, columns=[c for c in wanted if c in available])
    split_meta = read_metadata(split_path, ['molecule_id', 'split3', 'ik14'])
    train_meta = read_metadata(train_path, ['molecule_id', 'smiles', 'ik14'])
    split_meta['molecule_id'] = split_meta.molecule_id.astype(str)
    train_meta['molecule_id'] = train_meta.molecule_id.astype(str)
    held = split_meta.loc[split_meta.split3.ne('train')]
    held_train = train_meta.loc[train_meta.molecule_id.isin(held.molecule_id)]
    held_smiles = held_train.smiles.dropna().unique()
    keys = set()
    for s in held_smiles:
        mol = candidates.standardize_mol(s)
        if mol is not None:
            keys.add(evalkit.fast_key(mol))
        keys.add(evalkit.fast_key(s))
    for frame in (held, held_train):
        if 'ik14' in frame:
            keys.update(frame.ik14.dropna().astype(str))
    return {key for key in keys if key}

db_root = locate('db.parquet', DB_DIR).parent
pred_root = locate('fp_pred_val_model.npy', PRED_DIR).parent
pred_manifest = json.loads((pred_root / 'manifest.json').read_text())
assert pred_manifest['trained_on_split'] == 'train', 'Stage 05 split contract mismatch'
metadata = candidates.configure(db_root, exclusion_scope='train', row_group_cache=2)
# Compact Stage 04 datasets can omit build.sqlite.
if (db_root / 'build.sqlite').is_file():
    with candidates.open_sql(db_root / 'build.sqlite', readonly=True) as db:
        exclude_keys = candidates.validation_exclusions(db)
else:
    exclude_keys = heldout_exclusions(locate('splits.parquet', SPLIT_DIR),
                                      locate('train_clean.parquet', DATA_DIR))
print('Database:', db_root, 'held-out keys:', len(exclude_keys))

## Align observed query masses with prediction rows
Only metadata columns are read, never peak arrays or the complete compound DB.
Observed neutral masses are averaged across spectra per molecule, matching
Stage 04. No ground-truth formula or structure-derived mass is used to retrieve
validation candidates. ID files are authoritative for prediction alignment.

In [ ]:
def read_queries(path):
    columns = pq.ParquetFile(path).schema_arrow.names
    wanted = ['molecule_id', 'neutral_mass', 'precursor_mz', 'adduct', 'polarity']
    frame = pd.read_parquet(path, columns=[c for c in wanted if c in columns])
    frame['molecule_id'] = frame.molecule_id.astype(str)
    frame['neutral_mass'] = candidates.observed_mass(frame)
    # Conflicting spectrum adducts can produce extreme mass outliers.
    return frame.groupby('molecule_id', sort=False, as_index=False).neutral_mass.median()

train_path = locate('train_clean.parquet', DATA_DIR)
test_path = locate('test_clean.parquet', DATA_DIR)
split_path = locate('splits.parquet', SPLIT_DIR)
splits = pd.read_parquet(split_path, columns=['molecule_id', 'split3'])
splits['molecule_id'] = splits.molecule_id.astype(str)
splits = splits.drop_duplicates()
assert not splits.molecule_id.duplicated().any(), 'Conflicting splits'
train_queries = read_queries(train_path).merge(splits, on='molecule_id', how='left', validate='one_to_one')
assert train_queries.split3.notna().all(), 'Missing split assignments'
test_queries = read_queries(test_path)
test_queries['split3'] = 'test'

def load_split(name):
    ids = pd.read_parquet(pred_root / f'fp_pred_ids_{name}.parquet')
    ids['molecule_id'] = ids.molecule_id.astype(str)
    base = test_queries if name == 'test' else train_queries.loc[train_queries.split3.eq(name)]
    assert set(ids.molecule_id) == set(base.molecule_id), f'{name}: prediction ID mismatch'
    queries = ids[['molecule_id']].merge(base, on='molecule_id', how='left',
                                        sort=False, validate='one_to_one')
    fp = np.load(pred_root / f'fp_pred_{name}.npy', mmap_mode='r')
    elem = np.load(pred_root / f'elem_pred_{name}.npy', mmap_mode='r')
    maccs_path = pred_root / f'maccs_pred_{name}.npy'
    if not maccs_path.is_file():
        try:
            maccs_path = locate(f'maccs_pred_{name}.npy', PRED_DIR)
        except Exception:
            maccs_path = None
    maccs = np.load(maccs_path, mmap_mode='r') if maccs_path is not None and Path(maccs_path).is_file() else None
    if maccs is not None:
        print(f'{name}: loaded MACCS predictions {maccs.shape}')
    else:
        print(f'{name}: no MACCS predictions found; scoring with Morgan only')
    fprank.validate_inputs(queries, fp, elem, maccs=maccs)
    return queries, fp, elem, maccs

truth_by_smiles = {}
fast_truth_by_smiles = {}
def truth_for(queries, include_fast=False):
    frame = pd.read_parquet(train_path, columns=['molecule_id', 'smiles'])
    frame['molecule_id'] = frame.molecule_id.astype(str)
    frame = frame[frame.molecule_id.isin(queries.molecule_id)].dropna(subset=['smiles'])
    # A molecule_id may have multiple spectra with slight SMILES variations (kekule vs aromatic, etc.).
    # Pick the consensus (most frequent) valid SMILES per molecule_id to guarantee one structure per ID.
    smiles_counts = frame.groupby(['molecule_id', 'smiles']).size().reset_index(name='count')
    best_smiles = smiles_counts.sort_values(['molecule_id', 'count'], ascending=[True, False]).drop_duplicates(subset=['molecule_id'])
    truth = queries[['molecule_id']].merge(best_smiles[['molecule_id', 'smiles']], on='molecule_id', how='left', validate='one_to_one')
    assert truth.smiles.notna().all(), 'Missing truth SMILES for some validation queries'
    for s in tqdm(truth.smiles.unique(), desc='Unique truth keys'):
        if s not in truth_by_smiles:
            truth_by_smiles[s] = evalkit.canon_ik14(s)
        if include_fast and s not in fast_truth_by_smiles:
            mol = candidates.standardize_mol(s)
            fast_truth_by_smiles[s] = evalkit.fast_key(mol) if mol is not None else evalkit.fast_key(s)
    keys = [truth_by_smiles[s] for s in truth.smiles]
    assert all(keys), 'Invalid validation truth'
    if include_fast:
        return keys, [fast_truth_by_smiles[s] for s in truth.smiles]
    return keys

## Tune on val_model and freeze
The 3×3 ppm/temperature grid samples up to **5,000 val_model queries** with
NumPy RNG seed 42. Retrieve each sample's 20-ppm pool once, plus the wider
fallback when a smaller grid window is sparse. Compute all three temperature
feature sets together, retain only compact arrays, and run coordinate ascent
entirely in NumPy. No SQLite writes, Parquet reads, or canonicalization occur
during weight search. Runtime depends on candidate density and the host.

Tuning uses a fast_key/known-ik14 **MRR@25 proxy**, without tautomer deduplication;
full canonical deduplicated validation metrics are reported after freezing.
Three coordinate-ascent sweeps try 0, 0.5×, 0.75×, 1.25×, and 2× weights
(the last has a 0.1 floor). Equal improvements prefer smaller weight changes.
Report both overall proxy MRR and conditional MRR on the fixed subset whose
truth appears in the strict 20-ppm pool; fallback-only hits are excluded from
that conditional subset. Zero-hit queries remain in the overall denominator.

NB is divided by 4096 and negative element L1 by 10 before applying weights;
raw values are retained for Stage 09. Stage 05 probabilities have already had
its positive-weight correction applied. This stage tunes temperature, without
claiming to fit per-bit Platt slopes. Optional pre-fitted slopes/intercepts and
reliability weights are supported by the module and must come from val_model.
Provenance counts curated-source bits after removing the training bit.

In [ ]:
q_model, fp_model, elem_model, maccs_model = load_split('val_model')
truth_model, fast_truth_model = truth_for(q_model, include_fast=True)
q_model['truth_fast_key'] = fast_truth_model
tuning_cfg = dict(fprank.DEFAULT_CONFIG, exclude_keys=exclude_keys)
frozen = fprank.tune(q_model, fp_model, elem_model, truth_model, tuning_cfg,
                     sweeps=3, max_tune_queries=5000, maccs=maccs_model)
frozen['database_build_id'] = metadata['build_id']
frozen['element_order'] = list(fprank.ELEMENTS)
config_path = OUT / 'fprank_config.json'
config_path.write_text(json.dumps(frozen, indent=2))
# Reload the serialized contract; no further tuning after this point.
frozen = json.loads(config_path.read_text())
del fp_model, elem_model, maccs_model
gc.collect()
print('Frozen sampled tuning diagnostics:', {k: frozen[k] for k in
      ('tuning_queries', 'hit_queries_20ppm', 'oracle_20ppm_recall',
       'overall_proxy_mrr25', 'covered_hit_mrr25')})

## Stream ranked candidate tables
Every mass-window candidate is scored. If fewer than 25 survive filtering,
widen to the larger of ±50 ppm and ±0.05 Da. Keep a 256-row shortlist, then
canonicalize only until 25 unique canonical hits are collected. Fill ranks
26–100 using fast-key deduplication; their `canon_ik14` is null and
`identity_kind` is `fast_only`. Stage 09 must not treat those tail identities
as tautomer-canonical. Sparse windows may produce fewer than 100 rows.
If both windows are empty, score the nearest 50 eligible database rows so
test molecules retain predictions even outside the normal mass tolerance.
`scoring_source_mask` records the mask used for scoring; `source_mask` may
combine provenance when tautomer collisions are merged.
Lazy chemistry uses bounded memory caches and WAL/NORMAL SQLite persistence,
committing every 5,000 writes and at tuning/split boundaries, not per query.

Mass sorting improves mmap and row-group locality; fallback windows and lazy
fingerprint generation can still cause nonsequential reads. A 2048-row bit
batch occupies about 34 MB; total RSS also includes Python, Torch, database
row groups, and OS page cache, so a hard <2 GB total is not guaranteed.
`oracle_top100_recall` measures canonical-or-standardized-fast-key matches in the emitted top-100
shortlist, including fallback candidates; it is not full database membership.
`mrr25_all` includes all validation queries, including empty/missing shortlists.
`mrr25_given_in_pool` conditions on a top-100 truth-key match. Tail identities
remain fast/known keys, so this coverage diagnostic is a proxy for tautomer matches.

In [ ]:
summaries = {}
test_predictions = {}
val_model_truth = dict(zip(q_model.molecule_id, truth_model))
val_model_fast_truth = dict(zip(q_model.molecule_id, fast_truth_model))
output_schema = fprank.output_schema()
for name in ('val_model', 'val_rank', 'test'):
    queries, fp, elem, maccs = load_split(name)
    run_cfg = dict(frozen)
    if name != 'test':
        run_cfg['exclude_keys'] = exclude_keys
    path = OUT / f'cand_fp_{name}.parquet'
    writer, rows, found, rr = None, 0, set(), {}
    pending, pending_rows = [], 0
    if name == 'val_model':
        truth, fast_truth = val_model_truth, val_model_fast_truth
    elif name == 'val_rank':
        canonical_keys, fast_keys = truth_for(queries, include_fast=True)
        truth = dict(zip(queries.molecule_id, canonical_keys))
        fast_truth = dict(zip(queries.molecule_id, fast_keys))
    else:
        truth, fast_truth = {}, {}
    in_pool = dict.fromkeys(truth, False)
    rr = dict.fromkeys(truth, 0.0)
    try:
        iterator = fprank.iter_rank_candidates(queries, fp, elem, run_cfg, maccs_preds=maccs)
        for frame in tqdm(iterator, total=len(queries), desc=f'Ranking {name}'):
            if frame.empty:
                continue
            qid = frame.molecule_id.iloc[0]
            found.add(qid)
            table = pa.Table.from_pandas(frame, schema=output_schema, preserve_index=False, safe=True)
            if writer is None:
                writer = pq.ParquetWriter(path, output_schema, compression='zstd')
            pending.append(table)
            pending_rows += len(frame)
            if pending_rows >= 8192:
                writer.write_table(pa.concat_tables(pending))
                pending, pending_rows = [], 0
            rows += len(frame)
            if name == 'test':
                test_predictions[qid] = ';'.join(frame.smiles.head(25))
            else:
                hits = np.flatnonzero(frame.ik14.head(25).to_numpy() == truth[qid])
                rr[qid] = 1.0/(int(hits[0])+1) if len(hits) else 0.0
                in_pool[qid] = bool(
                    frame.ik14.head(100).eq(truth[qid]).any()
                    or frame.fast_key.head(100).eq(fast_truth[qid]).any()
                )
    finally:
        candidates.flush_cache()
        if writer is not None:
            if pending:
                writer.write_table(pa.concat_tables(pending))
            writer.close()
    if writer is None:
        raise RuntimeError(f'{name}: no candidates for any query; check database and masses')
    summaries[name] = dict(queries=len(queries), rows=rows, empty_queries=len(queries)-len(found))
    if name != 'test':
        covered, total_rr = sum(in_pool.values()), sum(rr.values())
        conditional = sum(rr[qid] for qid in truth if in_pool[qid]) / max(covered, 1)
        summaries[name].update(mrr25=total_rr/max(len(queries), 1),
            mrr25_all=total_rr/max(len(queries), 1), in_pool_queries=covered,
            oracle_top100_recall=covered/max(len(queries), 1),
            mrr25_given_in_pool=conditional, mrr25_when_in_top100=conditional)
    print(name, summaries[name])
    del fp, elem, maccs
    gc.collect()

## Submission validation and provenance
Empty mass windows use the nearest 50 eligible rows as a final coverage fallback.
A database with no eligible structures is an explicit error. Submission ID
completeness is checked before writing. Configuration stays frozen for val_rank
and test. Stage 10 must configure the same candidate DB before calling the API.

In [ ]:
submission = test_queries[['molecule_id']].copy()
submission['smiles'] = submission.molecule_id.map(test_predictions)
missing = submission.loc[submission.smiles.isna(), 'molecule_id'].tolist()
assert not missing, f'Coverage failure after nearest-row fallback: {missing[:20]}'
valid, errors, warnings, stats = evalkit.validate_submission(submission, test_queries.molecule_id)
assert valid, errors
submission.to_csv(OUT / 'submission_fprank.csv', index=False)
print(stats, warnings)

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

vm_ids = train_queries.loc[train_queries.split3.eq('val_model'), 'molecule_id'].tolist()
vr_ids = train_queries.loc[train_queries.split3.eq('val_rank'), 'molecule_id'].tolist()
manifest = dict(stage='fprank', tuned_on='val_model', trained_on_split='train',
                heldout_ids_hash=hashlib.sha256(''.join(sorted(vm_ids + vr_ids)).encode()).hexdigest(),
                exclusion_scope='train',
                database_build_id=metadata['build_id'], fp_spec=evalkit.FP_SPEC,
                inputs=dict(database=str(db_root), predictions=str(pred_root),
                            train=str(train_path), test=str(test_path), splits=str(split_path)),
                summaries=summaries, submission_validation=stats,
                elapsed_seconds=time.time()-STARTED,
                upstream_manifest_sha256=sha256(pred_root / 'manifest.json'),
                split_file_sha256=sha256(split_path),
                modules={n: sha256(WORK / n) for n in
                         ('evalkit.py', 'candidates.py', 'spec2fp.py', 'fprank.py')})
manifest['files'] = {p.name: dict(bytes=p.stat().st_size, sha256=sha256(p))
                     for p in sorted(OUT.iterdir()) if p.is_file() and p.name != 'manifest.json'}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Stage 06 complete:', OUT)